# E19 — Do TRAINING-SIDE methods reduce the day-shift gap? (IMU and audio-R0, separately)

**Context.** The setup:
- TII UAV Realistic Fault Dataset; 5-class propeller-fault severity.
- 95 missions after excluding the 4 corrupt ones. Classes [18, 19, 18, 20, 20]; days 2021-12-22 / 12-31 / 01-04 /
  01-05 / 01-10 with [10, 36, 17, 15, 17] missions (12-29 is merged into 12-31).

**Established facts:**
- Random-split results (~0.89) overstate performance. Leave-one-day-out gives ~0.82–0.83 (IMU 0.8347, audio R0 ~0.807).
- E16: no audio input representation fixes this.
- HTT/LACA and the fusion variants give no gain.

**E19 question.** Do *training-side* methods reduce the day-shift gap? Each modality is tested separately, with the
model and the protocol held fixed.

**Arms** (per modality; the same windows, missions, folds and seeds in every arm):

| arm | what changes vs A0 |
|---|---|
| **A0 BASE** | nothing: the reference training (must reproduce the reference per-fold F1 within seed noise; soft check) |
| **A1 AUG** | train-time augmentation only, fixed magnitudes chosen a priori; none at validation/test |
| **A2 CORAL** | A0 + CORAL loss (λ = 0.1, fixed) on the pooled embedding between training days, with a day-balanced sampler |
| **A3 DAYVAL** | A0, but the checkpoint is selected on a held-out TRAINING day (deterministic rule); the model is not refit |
| **A4 COMBO** | A1 + A2 |

**NOT ALLOWED in E19** (and not done here):
- per-mission or per-day normalization of test data (transductive: it leaks test-day statistics);
- test-day statistics of any kind;
- DANN, GroupDRO;
- the R6b channel subset;
- tuning any hyperparameter on the test day;
- changing the model.

**References (code copied verbatim where noted):**
- `RAN_TII_Audio_Only_Analysis_Mamba_ORIGINAL.ipynb`: setup, real mamba-ssm, constants, seeding, hooks, metadata
  audit, AudioBuffer parse cache, R0 windows and tokens, AudioMamba, bootstrap and metric helpers.
- `Ran_TII_IMU_Only_Analysis_Mamba.ipynb`: mission discovery, IMU parser and cache, data gate, IMU windows (2.56 s /
  1.28 s, Ts = 128 real samples chosen by index, 6 channels), IMUMamba, day-out folds.
- E16 notebook skeleton: flags, resumable loop, probability files, rule and analysis cells, generated summary, zip.

Deviations from the reference code are listed in the last cell of this notebook's build report, and inline as
`# E19:` comments.

In [ ]:
# ==============================================================================
# E19 RUN FLAGS (the only switches a user needs)
# ==============================================================================
SMOKE = False          # True: 1 fold (fold 1), seed 42, 2 epochs, every arm and modality -> *_smoke outputs, no analysis
RUN_SUBSET = None      # None = everything; or e.g. {"modalities": ["imu"], "arms": ["A0", "A1"]} (a printed subset)
SAVE_CKPT = False      # True: also save each run's best checkpoint (not needed for any result)

## 0. Setup (audio reference cells 0a–0d, verbatim except output names, v24/v28 constants and hook names)

In [ ]:
# ==============================================================================
# 0a. ENVIRONMENT, PATHS, DATASET
# ==============================================================================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must be set before the first cuBLAS call (determinism)

import ast
import copy
import csv
import gc
import glob
import hashlib
import json
import math
import random
import re
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
if IS_KAGGLE:
    BASE, OUT = Path("/kaggle/working/tii"), Path("/kaggle/working")
elif IS_COLAB:
    BASE, OUT = Path("/content/tii"), Path("/content")
else:
    BASE, OUT = Path.cwd() / "tii", Path.cwd() / "e19_outputs"
BASE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
ENV = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "local")

REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p: Path) -> bool:
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_attached = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _attached += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _attached:
    DATA_ROOT, DATA_SOURCE = _attached[0], "attached Kaggle dataset"
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"Cloning {REPO_URL} (--depth 1) into {_repo} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT, DATA_SOURCE = _repo / "Dataset", "git clone --depth 1"
assert _is_dataset_root(DATA_ROOT), f"{DATA_ROOT} does not contain class folders 0..4"

FIG_DIR = OUT / "figures"
CACHE_DIR = OUT / "e19_audio_cache"          # E19: own cache dir
PROBS_DIR = OUT / "e19_probs"
CKPT_DIR = OUT / "e19_checkpoints"
RESULTS_CSV = OUT / "e19_results.csv"
for _d in (FIG_DIR, CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV} | BASE {BASE} | outputs {OUT}")
print(f"Dataset: {DATA_ROOT} ({DATA_SOURCE})")
print(f"Figures -> {FIG_DIR} | cache -> {CACHE_DIR} | results -> {RESULTS_CSV} | probs -> {PROBS_DIR}")

In [ ]:
# ==============================================================================
# 0b. REAL mamba-ssm (no fallback). Installs only if it is not importable already.
# ==============================================================================
import importlib
import importlib.util

import torch


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Real mamba-ssm needs a GPU: enable a GPU accelerator (Kaggle: Settings -> "
                       "Accelerator -> GPU) and re-run. There is deliberately no CPU fallback in this notebook.")

if importlib.util.find_spec("mamba_ssm") is None:
    print(f"Installing mamba-ssm (torch {torch.__version__}, CUDA {torch.version.cuda}, python "
          f"{sys.version_info.major}.{sys.version_info.minor}) ...")
    _ok = (_pip(["packaging", "ninja"])
           and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output above). Check that internet is ON and a GPU "
                           "is attached, then restart the session and re-run. No fallback model is used.")

try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"mamba_ssm could not be imported ({exc!r}). Restart the kernel after installation and re-run; "
                       "no fallback model is used.") from exc
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} imported: {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | "
      f"torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 0c. CONSTANTS — every fixed value lives here
# ==============================================================================
SEEDS = [42, 123, 2024]

# --- AudioBuffer layout: 1536 = 256 frames x 6 interleaved slots; sample i -> frame i // 6, slot i % 6 ---
N_SLOTS = 6
DROP_SLOT = 5                                   # 0-based slot index 5 (the 6th slot): identically zero, dropped
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
N_CH = len(KEEP_SLOTS)                          # 5 audio channels
FRAME = 256                                     # frames per buffer = samples per channel per buffer
BUFFER_LEN = N_SLOTS * FRAME                    # 1536
N_RFFT = FRAME // 2 + 1                         # 129 per-channel rFFT bins (0..128)
KEEP_BINS = list(range(1, 17))                  # model input: per-channel bins 1..16 (fixed a priori)
N_FEAT = N_CH * len(KEEP_BINS)                  # 80

# --- windows (chunk counts only: no IMU, no time axis) ---
WINDOW_CHUNKS = 128
STRIDE_CHUNKS = 64
TA = 64                                         # tokens per window
TOKEN_POOL = WINDOW_CHUNKS // TA                # 2 consecutive chunks averaged per token
STEADY = (0.10, 0.90)                           # steady flight = 10%..90% of a mission's chunks (by position)

ASSUMED_CHANNEL_RATE_HZ = 16000                 # UNVERIFIED, used ONLY for secondary axis labels, never in any computation

# --- dataset facts (asserted in section 1) ---
N_CLASSES = 5
EXCLUDE_RATIO = 0.5                             # exclude if audio_start_offset / mission_duration >= this
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}        # the single remaining 12-29 mission joins the 12-31 group
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"

# --- model + training: transcribed from version 28/minor-project-v28.ipynb (config cell, UnimodalMamba, train_unimodal),
#     the audio arms A1/A2. E19: used unchanged; the v28 notebook is not read. ---
D_MODEL = 256
DROPOUT = 0.15
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496                 # 2 x 437,248 (real mamba_ssm, incl. the pre-norm LayerNorms)
EPOCHS = 20                                     # fixed; no early stopping in v28
BATCH_SIZE = 32
LEARNING_RATE = 1e-4                            # AdamW
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6                                   # CosineAnnealingLR(T_max=EPOCHS, eta_min=LR_MIN), stepped per epoch
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2
# checkpoint rule (v28): after every epoch, keep the state with the highest validation macro-F1 (strictly greater)

# --- splits ---
INNER_VAL_FRACTION, INNER_VAL_SEED = 0.15, 2027                    # v27 _split_inner_val (day-out protocol index 0)
INNER_VAL_FALLBACK_SEED = 2026                                     # used ONLY if v27's notebook cannot be read
V27_INNER_VAL_SHA1 = "58e3c037c513ace2"         # sha1[:16] of v27's _split_inner_val source (ast segment)

# --- analysis (fixed before seeing results) ---
N_PERMUTATIONS = 1000
PERM_SEED = 2029
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
TOP_CELLS = 10
SIGN_CONSISTENT_MIN_DAYS = 4
MIN_PER_CLASS_FOR_DAY_SIGN = 2
SPEARMAN_THRESHOLD = 0.7
PCA_BINS = list(range(1, 33))
EVIDENCE_K = [1, 2, 4, 8, 16, 32]

CHECK_DETERMINISM = False                       # E19: no hard determinism gate (mamba-ssm CUDA is non-deterministic); A0 vs reference is a soft check
DETERMINISM_EPOCHS = 3

HZ_PER_BIN = ASSUMED_CHANNEL_RATE_HZ / FRAME    # 62.5 Hz/bin — secondary axis labels ONLY
print(f"{N_SLOTS} slots x {FRAME} frames = {BUFFER_LEN} samples per buffer | channels kept {[s + 1 for s in KEEP_SLOTS]} "
      f"(slot {DROP_SLOT + 1} dropped) | model input {N_CH} ch x bins {KEEP_BINS[0]}-{KEEP_BINS[-1]} = {N_FEAT} features")
print(f"Windows: {WINDOW_CHUNKS} chunks, stride {STRIDE_CHUNKS}, {TA} tokens of {TOKEN_POOL} chunks | seeds {SEEDS}")

In [ ]:
# ==============================================================================
# 0c-IMU. IMU reference constants (Ran_TII_IMU_Only_Analysis_Mamba.ipynb, cell 'CONSTANTS', verbatim lines) +
#         a check that every setting SHARED with the audio reference is identical
# ==============================================================================
CHANNELS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
WIN_S = 2.56                     # WINDOW_SEC
STRIDE_S = 1.28                  # STEP_SEC
TS = 128                         # TS_SENSOR: samples per window, chosen evenly BY INDEX among the window's file-order samples
MIN_SAMPLES_PER_WINDOW = 10      # MIN_SENSOR_SAMPLES_PER_WINDOW
SENSOR_STD_EPS = 1e-6

IMU_REFERENCE_SHARED = {'SEEDS': [42, 123, 2024], 'N_CLASSES': 5, 'EXPECTED_EXCLUDED': {'rosbag2_2021_12_31-10_04_22', 'rosbag2_2021_12_31-09_58_13', 'rosbag2_2021_12_29-09_10_31', 'rosbag2_2021_12_31-10_12_31'}, 'EXPECTED_N_MISSIONS': 95, 'EXPECTED_CLASS_COUNTS': [18, 19, 18, 20, 20], 'DAYS': ['2021-12-22', '2021-12-31', '2022-01-04', '2022-01-05', '2022-01-10'], 'DAY_MERGE': {'2021-12-29': '2021-12-31'}, 'EXPECTED_DAY_COUNTS': [10, 36, 17, 15, 17], 'SMALL_FOLD_DAY': '2021-12-22', 'STEADY': (0.1, 0.9), 'D_MODEL': 256, 'DROPOUT': 0.15, 'MAMBA_NUM_BLOCKS': 2, 'MAMBA_D_STATE': 16, 'MAMBA_D_CONV': 2, 'MAMBA_EXPAND': 2, 'EXPECTED_MAMBA_PARAMS': 874496, 'EPOCHS': 20, 'BATCH_SIZE': 32, 'LEARNING_RATE': 0.0001, 'WEIGHT_DECAY': 0.0001, 'LR_MIN': 1e-06, 'GRAD_CLIP_NORM': 1.0, 'NUM_WORKERS': 2, 'INNER_VAL_FRACTION': 0.15, 'INNER_VAL_SEED': 2027, 'INNER_VAL_FALLBACK_SEED': 2026, 'V27_INNER_VAL_SHA1': '58e3c037c513ace2', 'BOOTSTRAP_RESAMPLES': 2000, 'BOOTSTRAP_SEED': 2028}
_diff = {k: (globals()[k], v) for k, v in IMU_REFERENCE_SHARED.items() if globals()[k] != v}
assert not _diff, f"IMU and audio reference settings differ: {_diff}"
print(f"IMU constants: windows {WIN_S} s / stride {STRIDE_S} s, Ts {TS}, channels {CHANNELS}; all "
      f"{len(IMU_REFERENCE_SHARED)} settings shared with the audio reference are identical")


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def find_file(fname, subdirs):
    cands = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        cands += [str(base / s / fname) for s in subdirs] + [str(base / fname)]
    return next((Path(c) for c in cands if Path(c).is_file()), None)

In [ ]:
# ==============================================================================
# 0c-E19. E19 CONSTANTS (arms, augmentation magnitudes, CORAL, rule, references) — ALL fixed a priori, never tuned
# ==============================================================================
MODALITY_ORDER = ["imu", "audio"]                       # IMU primary, audio-R0 secondary; never pooled
ARMS = ["A0", "A1", "A2", "A3", "A4"]
CANDIDATE_ARMS = ["A1", "A2", "A3", "A4"]               # K = 4
ARM_NAMES = {"A0": "BASE", "A1": "AUG", "A2": "CORAL", "A3": "DAYVAL", "A4": "COMBO (AUG + CORAL)"}
AUG_ARMS, CORAL_ARMS = {"A1", "A4"}, {"A2", "A4"}

# ---- A1 augmentation (training batches only) ----
IMU_AUG = {"scale_low": 0.9, "scale_high": 1.1,         # per-window, per-channel amplitude scale U(0.9, 1.1)
           "bias_sigma": 0.1,                           # per-window, per-channel bias N(0, 0.1 x train std)
           "noise_sigma": 0.05}                         # Gaussian noise N(0, 0.05 x train std); applied AFTER z-scoring
AUDIO_AUG = {"gain_low": 0.8, "gain_high": 1.25,        # per-window, per-channel amplitude gain g ~ logU(0.8, 1.25)
             "noise_frac": 0.05}                        # additive power-domain noise = 5% of per-channel train mean power
# ---- A2 CORAL ----
CORAL_LAMBDA = 0.1
# ---- A3 DAYVAL: held-out TRAINING day = the largest training day if it has >= 15 missions, else the second largest ----
A3_MIN_MISSIONS = 15

# ---- pre-registered rule (identical text in the markdown cell before the run cell and in e19_rule.txt) ----
MIN_DELTA = 0.02
PAIRED_CI = 99.0
MIN_FOLDS_SAME_SIGN = 4
RULE_TEXT = "PRE-REGISTERED DECISION RULE (E19; fixed before any E19 result; applied in code)\nPrimary metric: pooled out-of-fold WINDOW macro-F1 from seed-averaged probabilities (95 missions, day-out).\nSecondary: mission-level macro-F1 (argmax of the mean window probability), per-fold F1, worst-day F1.\nComparisons are made within ONE modality (IMU or audio-R0) and never pooled across modalities.\nCandidates: A1 AUG, A2 CORAL, A3 DAYVAL, A4 COMBO (K = 4), each against A0 BASE of the same modality.\nAn arm is CREDITED for a modality only if ALL hold vs A0:\n  1. pooled paired delta window macro-F1 >= +0.02 (seed noise floor ~0.016);\n  2. the 99% paired mission-cluster bootstrap CI of the delta excludes 0 (2000 resamples of missions within class,\n     seed 2028, the SAME resampled missions for both arms; 99% because K = 4);\n  3. delta > 0 in >= 4 of 5 folds (seed-averaged, per held-out day);\n  4. delta > 0 for all 3 seeds (each seed's own pooled out-of-fold F1);\n  5. worst-day F1 does not regress: min over folds of the arm's seed-averaged fold F1 >= A0's.\nWORSE (mirror): delta <= -0.02, the 99% CI lies entirely below 0, and delta < 0 in >= 4 of 5 folds.\nOtherwise: NOT CREDITED. Every arm is reported whatever the outcome; a null result is a valid outcome.\nDay gap closed (reported, not part of the rule): (arm - A0) / (random-split reference - A0), per modality."

# ---- references (context and the soft A0 check) ----
# Reference per-fold window macro-F1, mean over seeds 42/123/2024, from the reference runs' results CSVs
# (IMU: imu_only_results.csv; audio: audio_only_results.csv).
REF_FOLD_F1 = {"imu": [0.8324, 0.8489, 0.7260, 0.8316, 0.8129], "audio": [0.9000, 0.7488, 0.7415, 0.9189, 0.7846]}
REF_POOLED_F1 = {"imu": 0.8347, "audio": 0.8072}
REF_FOLD_TOL = 0.03
# Random-split reference for the "day gap closed" ratio: the project's random-split result (~0.89, stated in the E19
# brief; NOT re-measured here), used only as the gap's upper end.
RANDOM_SPLIT_REF = {"imu": 0.89, "audio": 0.89}
# Kaggle seconds per 20-epoch run in the reference runs (median), for the runtime estimate only.
REF_SECONDS_PER_RUN = {"imu": 62.0, "audio": 57.0}
BUDGET_HOURS = 4.0

# ---- run plan ----
if SMOKE:
    FOLDS_TO_RUN, SEEDS_TO_RUN, EPOCHS_TO_RUN = [1], [42], 2
    RESULTS_CSV = OUT / "e19_results_smoke.csv"
    PROBS_DIR = OUT / "e19_probs_smoke"
else:
    FOLDS_TO_RUN, SEEDS_TO_RUN, EPOCHS_TO_RUN = list(range(len(DAYS))), SEEDS, EPOCHS
MODS_TO_RUN = (RUN_SUBSET or {}).get("modalities", MODALITY_ORDER)
ARMS_TO_RUN = (RUN_SUBSET or {}).get("arms", ARMS)
assert set(MODS_TO_RUN) <= set(MODALITY_ORDER) and set(ARMS_TO_RUN) <= set(ARMS)
if "A0" not in ARMS_TO_RUN:
    print("NOTE: A0 is not in RUN_SUBSET: the paired analysis needs A0 runs (from this or an earlier session).")
PROBS_DIR.mkdir(parents=True, exist_ok=True)
(OUT / ("e19_rule_smoke.txt" if SMOKE else "e19_rule.txt")).write_text(RULE_TEXT + "\n", encoding="utf-8")
print(f"SMOKE={SMOKE} | modalities {MODS_TO_RUN} | arms {ARMS_TO_RUN} | folds {FOLDS_TO_RUN} | seeds {SEEDS_TO_RUN} | "
      f"epochs {EPOCHS_TO_RUN} | results -> {RESULTS_CSV.name} | probs -> {PROBS_DIR.name}/ | rule -> e19_rule.txt")

In [ ]:
# ==============================================================================
# 0d. SEEDING / DETERMINISM, FILE-ACCESS LOGGER, PASS/FAIL HOOKS, FIGURE HELPERS
# ==============================================================================
def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed: int) -> torch.Generator:
    """DataLoader shuffling uses its own generator, so the shuffle order depends only on the seed."""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id: int) -> None:
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
set_all_seeds(SEEDS[0])
print("Determinism settings: cudnn.deterministic=True, cudnn.benchmark=False, "
      "torch.use_deterministic_algorithms(True, warn_only=True) (warn_only because mamba-ssm's custom CUDA kernels are "
      f"outside PyTorch's deterministic registry), CUBLAS_WORKSPACE_CONFIG={os.environ['CUBLAS_WORKSPACE_CONFIG']}, "
      "per-run seeding of random/numpy/torch/CUDA, a seeded torch.Generator for DataLoader shuffling, and worker_init_fn.")

# ---- dataset-file access logger: EVERY dataset file read in this notebook goes through open_logged ----
ACCESS_KINDS = ("AudioBuffer:first_record", "AudioBuffer:full_parse", "SensorCombined:first_last_timestamp")
FILE_LOG = Counter()
FILE_LOG_PER_MISSION = {k: Counter() for k in ACCESS_KINDS}


def open_logged(path, kind, mode="r"):
    assert kind in ACCESS_KINDS, f"unknown access kind {kind}"
    FILE_LOG[kind] += 1
    FILE_LOG_PER_MISSION[kind][Path(path).parent.name] += 1
    return open(path, mode) if "b" in mode else open(path, mode, encoding="utf-8")


# ---- verification hooks ----
HOOKS = defaultdict(list)
HOOK_NAMES = {1: "mission/class/day counts + exclusion set", 2: "slot 6 identically zero", 3: "split leakage + train-only z-score",
              4: "windows: 128 chunks, within mission, tokens (64, 80)", 5: "identical window/mission ids across arms within a fold",
              6: "real mamba_ssm + params", 7: "augmentation never touches val/test",
              8: "A2/A4 batches contain >= 2 training days", 9: "A3 held-out training day logged; val has all 5 classes",
              10: "A0 vs reference per-fold F1 within 0.03 (SOFT)"}


def hook(n, ok, msg, stop=None):
    ok = bool(ok)
    HOOKS[n].append((ok, msg))
    print(f"[HOOK {n}] {'PASS' if ok else 'FAIL'} — {msg}")
    stop = (n <= 4) if stop is None else stop
    if not ok and stop:
        raise RuntimeError(f"HOOK {n} FAILED: {msg}. Stopping before anything is trained.")
    return ok


# ---- figures (palette: validated categorical order; sequential = one hue; diverging = two hues + neutral midpoint) ----
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CLASS_MARKERS = ["o", "s", "^", "D", "v"]
DAY_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
DAY_MARKERS = ["o", "s", "^", "D", "v"]
SEQ_CMAP, DIV_CMAP = "Blues", "RdBu_r"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "grid.linewidth": 0.6, "lines.linewidth": 1.5, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})
FREQ_LABEL = "per-channel FFT bin (1-128)"
HZ_LABEL = "approx. Hz if the per-channel rate were 16 kHz (unverified)"
FREQ_CAVEAT = ("Frequency axis = per-channel FFT bin of the de-interleaved channels. The Hz axis assumes a per-channel "
               "rate of 16 kHz, which is UNVERIFIED (the files carry no sample rate).")
FIGURES = []


def add_bin_axis(ax, axis="x"):
    """Secondary Hz axis for a per-channel-bin axis. Label only: HZ_PER_BIN is never used in a computation."""
    fwd, inv = (lambda b: b * HZ_PER_BIN), (lambda h: h / HZ_PER_BIN)
    if axis == "x":
        sec = ax.secondary_xaxis("top", functions=(fwd, inv))
        sec.set_xlabel(HZ_LABEL, fontsize=7)
    else:
        sec = ax.secondary_yaxis("right", functions=(fwd, inv))
        sec.set_ylabel(HZ_LABEL, fontsize=7)
    sec.tick_params(labelsize=7)
    ax._bin_caveat = True
    return sec


def save_fig(fig, name, caveat, freq_axes=()):
    """Every figure is saved as PNG with its caveat printed under it. Frequency figures must carry the Hz caveat."""
    assert caveat, f"{name}: every figure needs a caveat"
    freq_ok = all(getattr(ax, "_bin_caveat", False) for ax in freq_axes) and (not freq_axes or "UNVERIFIED" in caveat)
    fig.text(0.5, -0.01, caveat, ha="center", va="top", fontsize=8, color="#52514e", wrap=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    FIGURES.append({"name": name, "freq": bool(freq_axes), "freq_caveat_ok": freq_ok})
    print(f"  saved {FIG_DIR / (name + '.png')}" + ("" if freq_ok else "  <- MISSING frequency caveat"))


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

## 1. Audio: metadata audit, parse cache, integrity (audio reference 1a–1d, verbatim; own cache directory)
The metadata audit opens `SensorCombined` only to read its first and last `timestamp`. Only the R0 representation is
used: `pow[:, :, 1:17] ** 0.25` gives 80 features per chunk.

*E16's chunk-continuity hook is not repeated here.* It only decided whether E16's R7 representation (two chunks
joined in one FFT) could run, and E19 uses R0 only.

In [ ]:
# ==============================================================================
# 1a. METADATA AUDIT — recording_started_time vs. SensorCombined first/last timestamp; exclusion rule
# ==============================================================================
def mission_day(name: str) -> str:
    d = name[8:18].replace("_", "-")            # rosbag2_YYYY_MM_DD-... -> YYYY-MM-DD
    return DAY_MERGE.get(d, d)


def find_one(mdir: Path, pattern: str):
    hits = sorted(mdir.glob(pattern))
    return hits[0] if hits else None


ALL_DIRS, LABEL_OF_DIR = [], {}
for class_dir in sorted(DATA_ROOT.iterdir()):
    try:
        lab = int(class_dir.name)
    except ValueError:
        continue
    for mdir in sorted(class_dir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            ALL_DIRS.append(mdir)
            LABEL_OF_DIR[mdir] = lab
print(f"Mission folders found: {len(ALL_DIRS)} | per class {np.bincount(list(LABEL_OF_DIR.values()), minlength=N_CLASSES).tolist()}")

_TS_RE = re.compile(rb'"timestamp"\s*:\s*(\d+)')


def sensor_first_last_timestamp(path):
    """First and last `timestamp` (microseconds) of a SensorCombined file: the first line, then a backwards scan of the
    file tail. Regex on raw bytes: no IMU signal value is decoded."""
    with open_logged(path, "SensorCombined:first_last_timestamp", "rb") as f:
        first = None
        for line in f:
            m = _TS_RE.search(line)
            if m:
                first = int(m.group(1))
                break
        f.seek(0, 2)
        size = f.tell()
        block, last = min(size, 1 << 16), None
        while last is None:
            f.seek(size - block)
            lines = f.read(block).splitlines()
            for line in reversed(lines[1:] if block < size else lines):   # lines[0] may be a partial line
                m = _TS_RE.search(line)
                if m:
                    last = int(m.group(1))
                    break
            if last is None and block == size:
                break
            block = min(size, block * 4)
    return first, last


def audio_first_record(path):
    with open_logged(path, "AudioBuffer:first_record") as f:
        for line in f:
            if line.strip():
                return json.loads(line)
    return None


_meta = []
for mdir in ALL_DIRS:
    a_path, s_path = find_one(mdir, "*AudioBuffer*.jsonl"), find_one(mdir, "*SensorCombined*.jsonl")
    assert a_path is not None and s_path is not None, f"{mdir.name}: missing AudioBuffer or SensorCombined file"
    rec = audio_first_record(a_path)
    assert rec is not None and "data" in rec and "recording_started_time" in rec, f"{mdir.name}: unexpected AudioBuffer keys"
    t0, t1 = sensor_first_last_timestamp(s_path)
    rst = int(rec["recording_started_time"])
    dur = (t1 - t0) / 1e6
    off = (rst - t0) / 1e6
    _meta.append({"mission": mdir.name, "class": LABEL_OF_DIR[mdir], "day": mission_day(mdir.name),
                  "recording_started_time": rst, "imu_first_us": t0, "imu_last_us": t1, "audio_offset_s": off,
                  "mission_duration_s": dur, "offset_over_duration": off / dur})
META = pd.DataFrame(_meta).set_index("mission")

EXCLUDED = set(META.index[META.offset_over_duration >= EXCLUDE_RATIO])
print("\nExcluded (offset / duration >= 0.5):")
print(META.loc[sorted(EXCLUDED), ["class", "day", "audio_offset_s", "mission_duration_s", "offset_over_duration"]].to_string())
_kept_meta = META.drop(index=list(EXCLUDED))
print(f"\nMax offset/duration among KEPT missions: {_kept_meta.offset_over_duration.max():.3f} "
      f"({_kept_meta.offset_over_duration.idxmax()}) — below {EXCLUDE_RATIO}")
print(f"Audio start offset among kept missions: median {_kept_meta.audio_offset_s.median():.2f} s, "
      f"range {_kept_meta.audio_offset_s.min():.2f}–{_kept_meta.audio_offset_s.max():.2f} s")
EXCLUSION_OK = (EXCLUDED == EXPECTED_EXCLUDED) and bool((_kept_meta.offset_over_duration < EXCLUDE_RATIO).all())
print(f"Excluded set == the 4 expected missions and no other mission has ratio >= {EXCLUDE_RATIO}: {EXCLUSION_OK}")

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
axs[0].hist(_kept_meta.audio_offset_s, bins=30, color=CLASS_COLORS[0], edgecolor="white", label="kept (95)")
axs[0].hist(META.loc[sorted(EXCLUDED), "audio_offset_s"].clip(upper=_kept_meta.audio_offset_s.max() * 3), bins=10,
            color=CLASS_COLORS[1], edgecolor="white", label="excluded (4, clipped)")
axs[0].set_xlabel("audio start offset (s) = (recording_started_time − first IMU timestamp) / 1e6")
axs[0].set_ylabel("missions")
axs[0].legend()
axs[1].scatter(META.mission_duration_s, META.offset_over_duration.clip(-0.2, 2), s=14,
               c=[CLASS_COLORS[1] if n in EXCLUDED else CLASS_COLORS[0] for n in META.index])
axs[1].axhline(EXCLUDE_RATIO, color="#52514e", lw=1, ls="--")
axs[1].set_xlabel("mission duration (s, first->last IMU timestamp)")
axs[1].set_ylabel("offset / duration (clipped to [-0.2, 2])")
fig.suptitle("Audio start offset relative to the IMU log (metadata only)")
save_fig(fig, "1a_audio_offset_distribution",
         "Only timestamps are used: recording_started_time (AudioBuffer) and the first/last SensorCombined timestamp. "
         "Dashed line = exclusion threshold 0.5.")

KEPT = sorted((d for d in ALL_DIRS if d.name not in EXCLUDED), key=lambda d: d.name)
LABEL = {d: LABEL_OF_DIR[d] for d in KEPT}                      # keyed by Path (v27's inner-val rule expects this)
DAY = {d: mission_day(d.name) for d in KEPT}
NAME2DIR = {d.name: d for d in KEPT}
LABEL_N = {d.name: LABEL[d] for d in KEPT}
DAY_N = {d.name: DAY[d] for d in KEPT}
print("\nState: no IMU signal values are read anywhere in this notebook — SensorCombined files are opened only here, "
      "and only their first and last 'timestamp' fields are extracted.")

In [ ]:
# ==============================================================================
# 1b. PARSE EACH KEPT AudioBuffer FILE ONCE -> cache/<mission>.npz (skipped if the cache exists)
# ==============================================================================
def parse_and_cache(mdir: Path) -> str:
    cache = CACHE_DIR / f"{mdir.name}.npz"
    if cache.exists():
        return "cached"
    path = find_one(mdir, "*AudioBuffer*.jsonl")
    chunks, lens, rsts, n_bad = [], Counter(), set(), 0
    with open_logged(path, "AudioBuffer:full_parse") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                n_bad += 1
                continue
            x = rec.get("data")
            if x is None:
                n_bad += 1
                continue
            lens[len(x)] += 1
            chunks.append(x)
            rsts.add(rec.get("recording_started_time"))
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal} != {BUFFER_LEN}"
    assert len(rsts) == 1, f"{mdir.name}: recording_started_time is not constant ({len(rsts)} values)"
    kept = [x for x in chunks if len(x) == modal]
    raw = np.asarray(kept, dtype=np.int64)                      # (n, 1536)
    del chunks, kept
    n = len(raw)
    frames = raw.reshape(n, FRAME, N_SLOTS)                     # sample i -> frame i // 6, slot i % 6
    slot_vals = frames.reshape(-1, N_SLOTS).astype(np.float64)
    ch = frames[:, :, KEEP_SLOTS].astype(np.float64)
    ch = ch - ch.mean(axis=1, keepdims=True)                    # per-chunk mean removed, per channel
    rms = np.sqrt((ch ** 2).mean(axis=1)).astype(np.float32)    # (n, 5)
    pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)   # (n, 5, 129), rectangular window
    vmin, vmax = int(raw.min()), int(raw.max())
    tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
    np.savez(tmp, pow=pw, rms=rms, slot_std=slot_vals.std(axis=0), slot_mean=slot_vals.mean(axis=0),
             slot_zero=np.all(slot_vals == 0, axis=0), slot_absmax=np.abs(slot_vals).max(axis=0), n_chunks=n,
             mid_buffer=raw[n // 2].astype(np.int32), vmin=vmin, vmax=vmax, n_at_min=int((raw == vmin).sum()),
             n_at_max=int((raw == vmax).sum()), n_dropped=sum(v for k, v in lens.items() if k != modal),
             n_bad_lines=n_bad, modal_len=modal, recording_started_time=int(next(iter(rsts))))
    os.replace(tmp, cache)                                      # atomic: an interrupted parse never leaves a bad cache
    return "parsed"


_t0 = time.time()
PARSE_STATUS = {}
for i, mdir in enumerate(KEPT, 1):
    PARSE_STATUS[mdir.name] = parse_and_cache(mdir)
    if i % 10 == 0 or i == len(KEPT):
        print(f"  {i}/{len(KEPT)} missions | parsed {sum(v == 'parsed' for v in PARSE_STATUS.values())}, cached "
              f"{sum(v == 'cached' for v in PARSE_STATUS.values())} | {time.time() - _t0:.0f} s")

# ---- ONE load pass over the cache: keep only the compact derived arrays used below ----
MISS = {}
for mdir in KEPT:
    z = np.load(CACHE_DIR / f"{mdir.name}.npz")
    pw = z["pow"]
    n = len(pw)
    a, b = int(STEADY[0] * n), int(STEADY[1] * n)
    MISS[mdir.name] = {
        "n_chunks": n, "rms": z["rms"], "steady_pow": pw[a:b].astype(np.float64).mean(axis=0),   # (5, 129)
        "feats": (pw[:, :, KEEP_BINS] ** 0.25).reshape(n, N_FEAT).astype(np.float32),        # sqrt|rFFT|, channel-major
        "band": pw[:, :, KEEP_BINS].sum(axis=2),                                               # (n, 5) power in bins 1-16
        "slot_std": z["slot_std"], "slot_mean": z["slot_mean"], "slot_zero": z["slot_zero"], "mid_buffer": z["mid_buffer"],
        "vmin": int(z["vmin"]), "vmax": int(z["vmax"]), "n_at_min": int(z["n_at_min"]), "n_at_max": int(z["n_at_max"]),
        "n_dropped": int(z["n_dropped"]), "n_bad_lines": int(z["n_bad_lines"]), "modal_len": int(z["modal_len"]),
        "finite": bool(np.isfinite(pw).all()), "rst_cache": int(z["recording_started_time"])}
    del pw, z
_dropped = sum(v["n_dropped"] for v in MISS.values())
print(f"\nModal buffer length = {BUFFER_LEN} in every mission: {all(v['modal_len'] == BUFFER_LEN for v in MISS.values())} | "
      f"non-modal chunks dropped: {_dropped} (in {sum(v['n_dropped'] > 0 for v in MISS.values())} missions) | "
      f"undecodable lines: {sum(v['n_bad_lines'] for v in MISS.values())}")
assert all(v["modal_len"] == BUFFER_LEN for v in MISS.values())
assert all(v["rst_cache"] == META.loc[k, "recording_started_time"] for k, v in MISS.items()), "cache/metadata mismatch"
print(f"Cache: {len(MISS)} missions, {sum(v['n_chunks'] for v in MISS.values()):,} chunks | load pass {time.time() - _t0:.0f} s total")

In [ ]:
# ==============================================================================
# 1c. COUNTS — missions, classes, days (HOOK 1)
# ==============================================================================
_cls = np.bincount([LABEL[d] for d in KEPT], minlength=N_CLASSES).tolist()
_days = [sum(DAY[d] == day for d in KEPT) for day in DAYS]
DAY_CLASS = pd.crosstab(pd.Series([DAY[d] for d in KEPT], name="day"), pd.Series([LABEL[d] for d in KEPT], name="class"))
DAY_CLASS = DAY_CLASS.reindex(index=DAYS, columns=range(N_CLASSES), fill_value=0)
print(f"Missions: {len(KEPT)} | per class {_cls} | per day {dict(zip(DAYS, _days))}")
print(f"Merged into 2021-12-31: {[d.name for d in KEPT if d.name[8:18] == '2021_12_29']}")
print("\nDay x class (missions):")
print(DAY_CLASS.assign(total=DAY_CLASS.sum(axis=1)).to_string())
print(f"Every class on every day: {bool((DAY_CLASS.values > 0).all())}")
hook(1, len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and EXCLUSION_OK and _days == EXPECTED_DAY_COUNTS
     and bool((DAY_CLASS.values > 0).all()),
     f"{len(KEPT)} missions, classes {_cls}, excluded set == the 4 names: {EXCLUSION_OK}, days {_days} "
     f"(expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, True, {EXPECTED_DAY_COUNTS})")

In [ ]:
# ==============================================================================
# 1d. SIX-SLOT VERIFICATION across all 95 missions (HOOK 2)
# ==============================================================================
_std = np.stack([MISS[d.name]["slot_std"] for d in KEPT])        # (95, 6)
_mean = np.stack([MISS[d.name]["slot_mean"] for d in KEPT])
_zero = np.stack([MISS[d.name]["slot_zero"] for d in KEPT])
_slot_tab = pd.DataFrame({"slot": [f"slot {s + 1}" for s in range(N_SLOTS)], "mean std": _std.mean(0).round(2),
                          "median std": np.median(_std, 0).round(2), "mean DC (slot mean)": _mean.mean(0).round(2),
                          "missions identically zero": _zero.sum(0),
                          "missions where largest std": np.bincount(_std.argmax(1), minlength=N_SLOTS)}).set_index("slot")
print("Per-slot statistics over all samples of each mission, then across the 95 missions:")
print(_slot_tab.to_string())
n_zero6 = int(_zero[:, DROP_SLOT].sum())
n_slot1_max = int((_std.argmax(1) == 0).sum())
print(f"\nSlot 6 identically zero in {n_zero6}/{len(KEPT)} missions | slot 1 has the largest std in {n_slot1_max}/{len(KEPT)}")

_vmin = min(v["vmin"] for v in MISS.values())
_vmax = max(v["vmax"] for v in MISS.values())
_at_ext = [(k, v["n_at_min"], v["n_at_max"]) for k, v in MISS.items() if v["vmin"] == _vmin or v["vmax"] == _vmax]
print(f"\nClipping/saturation: observed global sample range [{_vmin}, {_vmax}] "
      f"(int16 full scale is [-32768, 32767]: {'reached' if _vmin <= -32768 or _vmax >= 32767 else 'NOT reached'})")
print(f"  missions touching the global extreme: {len(_at_ext)}; samples sitting exactly at their own min/max per mission "
      f"(median over missions): {np.median([v['n_at_min'] + v['n_at_max'] for v in MISS.values()]):.0f} "
      f"(a large count at the extreme would indicate clipping)")
print(f"NaN/Inf in any cached spectrum: {not all(v['finite'] for v in MISS.values())}")

_nch = pd.DataFrame({"mission": [d.name for d in KEPT], "class": [LABEL[d] for d in KEPT], "day": [DAY[d] for d in KEPT],
                     "chunks": [MISS[d.name]["n_chunks"] for d in KEPT]})
print("\nChunks per mission: " + _nch.chunks.describe()[["min", "25%", "50%", "75%", "max"]].round(0).to_string().replace("\n", " | "))
print("Chunks per class:")
print(_nch.groupby("class").chunks.agg(["count", "sum", "median", "min", "max"]).to_string())
_imp_ms = [(META.loc[d.name, "imu_last_us"] - META.loc[d.name, "recording_started_time"]) / 1e3 / MISS[d.name]["n_chunks"]
           for d in KEPT]
print(f"\nDESCRIPTIVE ONLY (not used anywhere): if audio ran from recording_started_time to the last IMU timestamp, one "
      f"buffer would last {np.percentile(_imp_ms, 5):.1f}–{np.percentile(_imp_ms, 95):.1f} ms (5th–95th pct, median "
      f"{np.median(_imp_ms):.1f} ms). 16 ms (16 kHz per channel) is consistent with this, but it is unverified. A 48 kHz "
      f"mono assumption would imply {BUFFER_LEN / 48:.0f} ms.")
hook(2, n_zero6 == len(KEPT) == EXPECTED_N_MISSIONS, f"slot 6 identically zero in {n_zero6}/{len(KEPT)} missions")

In [ ]:
# ==============================================================================
# 4. WINDOWS AND TOKENS (HOOK 4)
# ==============================================================================
NAMES = [d.name for d in KEPT]                                   # audio reference cell 2d
_X, _wm = [], []
for n in NAMES:
    Fm = MISS[n]["feats"]
    N = len(Fm)
    for i, s in enumerate(range(0, N - WINDOW_CHUNKS + 1, STRIDE_CHUNKS)):
        _X.append(Fm[s:s + WINDOW_CHUNKS].reshape(TA, TOKEN_POOL, N_FEAT).mean(axis=1))
        rel = (s + WINDOW_CHUNKS / 2) / N
        _wm.append({"window_id": f"{n}#{i:04d}", "mission": n, "cls": LABEL_N[n], "day": DAY_N[n], "start": s,
                    "end": s + WINDOW_CHUNKS, "mission_chunks": N, "rel_centre": rel,
                    "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady")})
X_ALL = np.stack(_X).astype(np.float32)
WMETA = pd.DataFrame(_wm)
del _X, _wm
WIN_ID2ROW = {w: i for i, w in enumerate(WMETA.window_id)}

_len_ok = bool(((WMETA.end - WMETA.start) == WINDOW_CHUNKS).all())
_inside = bool(((WMETA.start >= 0) & (WMETA.end <= WMETA.mission_chunks)).all())
_one_mission = WMETA.groupby("window_id").mission.nunique().max() == 1 and WMETA.window_id.is_unique
_shape_ok = X_ALL.shape[1:] == (TA, N_FEAT)
_finite = bool(np.isfinite(X_ALL).all())
print(f"Tokens: X_ALL {X_ALL.shape} (windows, tokens, features) | NaN/Inf present: {not _finite}")
print(f"Window = {WINDOW_CHUNKS} chunks ≈ {WINDOW_CHUNKS * FRAME / ASSUMED_CHANNEL_RATE_HZ:.2f} s ONLY IF a buffer is "
      f"{1000 * FRAME / ASSUMED_CHANNEL_RATE_HZ:.0f} ms (unverified)")
print("\nWindows per class x phase:")
print(pd.crosstab(WMETA.cls, WMETA.phase).reindex(columns=["start", "steady", "end"]).assign(total=lambda t: t.sum(1)).to_string())
print(f"Windows per mission: median {WMETA.groupby('mission').size().median():.0f}, "
      f"min {WMETA.groupby('mission').size().min()}, max {WMETA.groupby('mission').size().max()}")
_fr = X_ALL.reshape(-1, N_CH, len(KEEP_BINS))
print("\nFeature ranges (sqrt|rFFT|, before z-scoring) per channel: " + " | ".join(
    f"ch{c + 1} min {_fr[:, c].min():.2f} median {np.median(_fr[::7, c]):.2f} max {_fr[:, c].max():.2f}" for c in range(N_CH)))
del _fr
hook(4, _len_ok and _inside and _one_mission and _shape_ok and _finite and len(WMETA) == len(X_ALL),
     f"{len(WMETA)} windows: every window has {WINDOW_CHUNKS} chunks ({_len_ok}), lies inside one mission ({_inside and _one_mission}), "
     f"tokens {X_ALL.shape[1:]} ({_shape_ok}), finite ({_finite})")

## 2. IMU: missions, parser, cache, data gate, windows (IMU reference, verbatim; own cache directory)
Samples stay in **file order**. Windows are 2.56 s with a 1.28 s stride. Ts = 128 real samples are chosen evenly
**by index** (nothing is interpolated), across 6 channels.

In [ ]:
# Discover missions and SensorCombined files; apply the project's 4-mission exclusion
def mission_day(name):
    d = name[8:18].replace("_", "-")          # rosbag2_YYYY_MM_DD-... -> YYYY-MM-DD
    return DAY_MERGE.get(d, d)


mission_rows = []
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not class_dir.is_dir() or not class_dir.name.isdigit():
        continue
    cls = int(class_dir.name)
    for mission_dir in sorted(class_dir.iterdir()):
        if not mission_dir.is_dir() or mission_dir.name.startswith("."):
            continue
        candidates = sorted(mission_dir.glob("*-SensorCombined.jsonl"))
        mission_rows.append({"mission": mission_dir.name, "class": cls, "day": mission_day(mission_dir.name),
                             "path": str(candidates[0]) if candidates else None})

all_missions = pd.DataFrame(mission_rows)
missions = all_missions[~all_missions["mission"].isin(EXPECTED_EXCLUDED)].reset_index(drop=True)
print(f"mission folders: {len(all_missions)} | excluded (project-wide list): {sorted(EXPECTED_EXCLUDED)}")
print("missions kept:", len(missions))
print(missions.groupby("class").size().rename("missions").to_frame().T.to_string())
print("\nday x class:")
print(pd.crosstab(missions["day"], missions["class"]).reindex(DAYS).to_string())
missions.head()

In [ ]:
# Parser functions (IMU reference cells 'unwrap_message' and 'Full IMU parser', verbatim)
def unwrap_message(obj):
    # Handle common ROS/PX4 JSONL wrappers.
    if isinstance(obj, dict):
        for key in ("msg", "message", "data"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def get_vec(d, candidates):
    for key in candidates:
        if key in d:
            v = d[key]
            if isinstance(v, (list, tuple)) and len(v) >= 3:
                return np.asarray(v[:3], dtype=np.float64)
        # Some JSON encodings may flatten [0], [1], [2].
        vals = []
        ok = True
        for i in range(3):
            k = key.format(i=i)
            if k in d:
                vals.append(d[k])
            else:
                ok = False
                break
        if ok:
            return np.asarray(vals, dtype=np.float64)
    return None


def parse_imu_file(path):
    """Rows [timestamp_us, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z] in FILE ORDER (never sorted here)."""
    rows = []
    bad = 0
    with open(path, "r") as f:
        for line in f:
            try:
                obj = unwrap_message(json.loads(line))
                ts = obj.get("timestamp", obj.get("time_stamp", None))
                acc = get_vec(obj, [
                    "accelerometer_m_s2",
                    "accelerometer_m_s2[{i}]",
                    "accelerometer",
                    "acceleration",
                ])
                gyro = get_vec(obj, [
                    "gyro_rad",
                    "gyro_rad[{i}]",
                    "gyro",
                    "angular_velocity",
                ])
                if ts is None or acc is None or gyro is None:
                    bad += 1
                    continue
                rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
            except Exception:
                bad += 1

    arr = np.asarray(rows, dtype=np.float64).reshape(-1, 7)
    return arr, bad

In [ ]:
OUT_DIR = OUT / "e19_imu_cache"                   # E19: own IMU cache directory (reference: WORK / 'imu_only')
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Parse/cache all kept missions. The cache stores FILE ORDER; the sorted copy is derived on load.
cache_dir = OUT_DIR / "cache_fileorder"        # new name: an older cache in OUT_DIR/cache held SORTED arrays
cache_dir.mkdir(exist_ok=True)

imu_data = {}
integrity = []

for i, r in missions.iterrows():
    cache_path = cache_dir / f"{r['mission']}.npz"
    if r["path"] is None:
        t, x, bad = np.empty(0), np.empty((0, 6)), 0
    elif cache_path.exists():
        z = np.load(cache_path)
        x = z["x"]
        t = z["t"]
        bad = int(z["bad"])
    else:
        raw, bad = parse_imu_file(Path(r["path"]))
        t, x = raw[:, 0], raw[:, 1:]
        np.savez_compressed(cache_path, t=t, x=x, bad=bad)

    order = np.argsort(t, kind="stable")
    imu_data[r["mission"]] = {"t": t, "x": x, "t_sorted": t[order], "x_sorted": x[order],
                              "class": int(r["class"]), "day": r["day"]}

    dt = np.diff(t) / 1e6                       # FILE ORDER: an inversion shows up as a negative interval
    integrity.append({
        "mission": r["mission"],
        "class": int(r["class"]),
        "day": r["day"],
        "n": len(t),
        "duration_s": (t[-1]-t[0])/1e6 if len(t)>1 else np.nan,
        "median_dt_ms": np.median(dt)*1e3 if len(dt) else np.nan,
        "p01_dt_ms": np.percentile(dt,1)*1e3 if len(dt) else np.nan,
        "p99_dt_ms": np.percentile(dt,99)*1e3 if len(dt) else np.nan,
        "max_dt_ms": np.max(dt)*1e3 if len(dt) else np.nan,
        "n_negative_dt": int((dt < 0).sum()),
        "bad_lines": bad,
        "finite": bool(len(t) > 0 and np.isfinite(t).all() and np.isfinite(x).all()),
    })

integrity = pd.DataFrame(integrity)
print(integrity.groupby("class")[["n","duration_s","median_dt_ms","max_dt_ms"]].median())
print("total bad lines:", integrity.bad_lines.sum())

In [ ]:
# ==============================================================================
# VALIDATION GATE — part A (data)
# ==============================================================================
GATE = []


def gate(name, ok, detail=""):
    ok = bool(ok)
    GATE.append((name, ok, detail))
    print(f"[GATE] {'PASS' if ok else 'FAIL'} — {name}" + (f" | {detail}" if detail else ""))
    return ok


_cls = np.bincount(missions["class"], minlength=N_CLASSES).tolist()
_days = [int((missions["day"] == d).sum()) for d in DAYS]
gate("mission and class counts", len(missions) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS,
     f"{len(missions)} missions, per class {_cls} (expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS})")
gate("day counts", _days == EXPECTED_DAY_COUNTS and set(missions["day"]) == set(DAYS),
     f"{dict(zip(DAYS, _days))} (expected {EXPECTED_DAY_COUNTS}; 12-29 merged into 12-31)")
_bad_finite = [m for m in missions["mission"] if not (len(imu_data[m]["t"]) > 0 and np.isfinite(imu_data[m]["t"]).all()
                                                       and np.isfinite(imu_data[m]["x"]).all())]
gate("every mission's parsed array is non-empty and finite (checked per mission)", not _bad_finite,
     f"{len(missions) - len(_bad_finite)}/{len(missions)} OK" + (f"; FAILING: {_bad_finite}" if _bad_finite else ""))
_folders = set(all_missions["mission"])
_kept = set(missions["mission"])
_excl = _folders - _kept
gate("no mission excluded that shouldn't be, and no unexpected mission missing",
     _excl == EXPECTED_EXCLUDED and EXPECTED_EXCLUDED <= _folders and all_missions.loc[
         all_missions.mission.isin(_kept), "path"].notna().all(),
     f"excluded = {sorted(_excl)}; all 4 expected folders present: {EXPECTED_EXCLUDED <= _folders}; "
     f"every kept mission has a SensorCombined file: {all_missions.loc[all_missions.mission.isin(_kept), 'path'].notna().all()}")
gate("window ids/count per fold match across protocols", True, "N/A — day-out only (no matched-random protocol here)")
print("[GATE] determinism check -> part B, immediately before the day-out runs (needs the model and folds)")
if not all(ok for _, ok, _ in GATE):
    loud("VALIDATION GATE (part A) FAILED:\n" + "\n".join(f"{n}: {d}" for n, ok, d in GATE if not ok)
         + "\nStopping before section 3. Fix the data/paths and re-run.")
    raise RuntimeError("Validation gate (part A) failed — see the FAIL lines above.")
print("Validation gate part A: all PASS")

In [ ]:
windows=[]
for _,r in missions.iterrows():
    m=r["mission"]; d=imu_data[m]
    t=(d["t"]-d["t"][0])/1e6          # FILE ORDER, seconds relative to the first sample (v24)
    x=d["x"]
    if len(t)<2 or t[-1]<WIN_S:
        continue
    dur=t[-1]

    starts=np.arange(0, dur-WIN_S+1e-9, STRIDE_S)
    i=0
    for s in starts:
        e=s+WIN_S
        idx=np.where((t>=s)&(t<=e))[0]                     # file-order positions inside the window
        if len(idx)<MIN_SAMPLES_PER_WINDOW:
            continue
        pick=idx[np.linspace(0,len(idx)-1,TS).round().astype(int)]   # v24: TS real samples, evenly by index
        rel=(s+WIN_S/2)/dur
        windows.append({"window_id":f"{m}#{i:04d}","mission":m,"class":int(r["class"]),"day":r["day"],
                        "start_s":float(s),"phase":"start" if rel<STEADY[0] else ("end" if rel>STEADY[1] else "steady"),
                        "n_samples":len(idx),"x":x[pick].astype(np.float32)})
        i+=1

print("windows:",len(windows))
_wpm = Counter(w["mission"] for w in windows)
assert len(_wpm) == len(missions), f"{len(missions) - len(_wpm)} missions produced no window"
assert all(w["x"].shape == (TS, 6) and np.isfinite(w["x"]).all() for w in windows)
assert len({w["window_id"] for w in windows}) == len(windows)
print(f"every window: shape ({TS}, 6), finite, one mission; windows per mission median {np.median(list(_wpm.values())):.0f} "
      f"(min {min(_wpm.values())}, max {max(_wpm.values())}); samples per window median "
      f"{np.median([w['n_samples'] for w in windows]):.0f}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

W=np.stack([w["x"] for w in windows]).astype(np.float32)
y=np.array([w["class"] for w in windows],dtype=np.int64)
groups=np.array([w["mission"] for w in windows])
days=np.array([w["day"] for w in windows])
phases=np.array([w["phase"] for w in windows])
window_ids=np.array([w["window_id"] for w in windows])
starts_s=np.array([w["start_s"] for w in windows])

print(W.shape, y.shape)
print(pd.crosstab(pd.Series(days, name="day"), pd.Series(y, name="class")).reindex(DAYS).to_string())

In [ ]:
# ==============================================================================
# 2b. E19: both modalities must cover the SAME 95 missions (the folds are shared)
# ==============================================================================
_imu_set, _aud_set = set(missions["mission"]), set(NAMES)
assert _imu_set == _aud_set and len(_imu_set) == EXPECTED_N_MISSIONS, (len(_imu_set), len(_aud_set), _imu_set ^ _aud_set)
assert set(groups) == _imu_set and set(WMETA.mission) == _aud_set, "a mission produced no window"
MISSION_DAY = {m: d for m, d in zip(missions["mission"], missions["day"])}
MISSION_CLASS = {m: int(c) for m, c in zip(missions["mission"], missions["class"])}
assert all(MISSION_DAY[n] == DAY_N[n] and MISSION_CLASS[n] == LABEL_N[n] for n in NAMES), "day/class labels differ"
print(f"Both modalities: the same {len(_imu_set)} missions with identical class and day labels | IMU windows {len(W)} | "
      f"audio windows {len(X_ALL)}")

## 3. Day-out folds (IMU reference cell "DAY-OUT FOLDS", verbatim; shared by both modalities)
- Fold k tests on day k.
- Inner validation holds out ~15% of each class's training missions. The rule is v27's `_split_inner_val` if
  `minor-project-v27.ipynb` is attached under `/kaggle/input`; otherwise the fixed seed-2026 fallback. **The rule that
  ran is printed.**
- The reference audio run behind 0.8072 used the fallback, so attaching v27 changes A0's splits relative to the
  references.

In [ ]:
# ==============================================================================
# DAY-OUT FOLDS — identical to the audio notebook (fold k = DAYS[k]; v27 inner-validation rule, else seed 2026)
# ==============================================================================
def extract_function(nb_path,fname):
    try:
        nb=json.loads(Path(nb_path).read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"  could not read {nb_path}: {exc!r}")
        return None
    for cell in nb.get("cells",[]):
        if cell.get("cell_type")!="code":
            continue
        src=cell["source"] if isinstance(cell["source"],str) else "".join(cell["source"])
        src="\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!","%")))
        try:
            tree=ast.parse(src)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node,ast.FunctionDef) and node.name==fname:
                return ast.get_source_segment(src,node)
    return None


LABEL={Path(m):int(c) for m,c in zip(missions["mission"],missions["class"])}   # v27's rule keys missions by Path (.name)
_p27=find_file("minor-project-v27.ipynb",["version 27","Version 27"])
_src27=extract_function(_p27,"_split_inner_val") if _p27 else None
if _src27 is not None and hashlib.sha1(_src27.encode()).hexdigest()[:16]==V27_INNER_VAL_SHA1:
    _ns27={"np":np,"N_CLASSES":N_CLASSES,"LABEL":LABEL,"INNER_VAL_SEED":INNER_VAL_SEED,"INNER_VAL_FRACTION":INNER_VAL_FRACTION}
    exec(_src27,_ns27)
    inner_val=lambda pool,k: _ns27["_split_inner_val"](pool,0,k)
    INNER_VAL_RULE=f"v27 _split_inner_val (from {_p27}; seed [{INNER_VAL_SEED}, 0, fold]) — same as the audio notebook"
else:
    loud(("v27 notebook not found" if _p27 is None else f"v27 _split_inner_val unreadable/fingerprint mismatch at {_p27}")
         + f"\nFALLBACK inner validation: per-class ~15% (min 1) mission hold-out, fixed seed {INNER_VAL_FALLBACK_SEED} "
           "(the audio notebook's fallback, identical code).")

    def inner_val(pool,k):
        rng=np.random.default_rng([INNER_VAL_FALLBACK_SEED,k])
        train,val=[],[]
        for c in range(N_CLASSES):
            ms=sorted((m for m in pool if LABEL[m]==c),key=lambda m:m.name)
            perm=rng.permutation(len(ms))
            n_val=max(1,int(round(INNER_VAL_FRACTION*len(ms))))
            val+=[ms[i] for i in perm[:n_val]]
            train+=[ms[i] for i in perm[n_val:]]
        return train,val

    INNER_VAL_RULE=f"FALLBACK per-class hold-out, seed {INNER_VAL_FALLBACK_SEED} (same as the audio notebook's fallback)"
print("Inner-validation rule:",INNER_VAL_RULE)

FOLDS={}
_all=set(missions["mission"])
for k,day in enumerate(DAYS):
    test=sorted(missions.loc[missions["day"]==day,"mission"])
    tr,va=inner_val([Path(m) for m in sorted(_all-set(test))],k)
    FOLDS[k]={"day":day,"train":sorted(p.name for p in tr),"val":sorted(p.name for p in va),"test":test}
    f=FOLDS[k]
    assert not (set(f["train"])&set(f["val"])) and not (set(f["train"]+f["val"])&set(test))
    assert set(f["train"]+f["val"]+test)==_all
    assert day not in {imu_data[m]["day"] for m in f["train"]+f["val"]}, f"fold {k}: test day in training"
assert sorted(m for f in FOLDS.values() for m in f["test"])==sorted(_all)

print(f"{'fold':<5}{'test day':<12}{'missions tr/val/test':<22}{'test missions per class':<26}{'windows tr/val/test'}")
for k,f in FOLDS.items():
    per_class=np.bincount([imu_data[m]["class"] for m in f["test"]],minlength=N_CLASSES).tolist()
    n_w="/".join(str(int(np.isin(groups,f[s]).sum())) for s in ("train","val","test"))
    print(f"{k:<5}{f['day']:<12}{'/'.join(str(len(f[s])) for s in ('train','val','test')):<22}{str(per_class):<26}{n_w}"
          + ("   <- SMALL fold (10 missions)" if f["day"]==SMALL_FOLD_DAY else ""))

In [ ]:
# ==============================================================================
# 3b. SPLIT HOOK 3: no test-day mission in train/val; inner validation has all 5 classes; per-fold mission hashes
# ==============================================================================
_ok3 = True
for k, f in FOLDS.items():
    _ok3 &= f["day"] not in {MISSION_DAY[m] for m in f["train"] + f["val"]}
    _ok3 &= not (set(f["train"] + f["val"]) & set(f["test"])) and not (set(f["train"]) & set(f["val"]))
    _ok3 &= {MISSION_CLASS[m] for m in f["val"]} == set(range(N_CLASSES))
    _ok3 &= {MISSION_CLASS[m] for m in f["train"]} == set(range(N_CLASSES))
FOLD_HASH = {k: hashlib.sha1(json.dumps({s: f[s] for s in ("train", "val", "test")}).encode()).hexdigest()[:12]
             for k, f in FOLDS.items()}
hook(3, _ok3, "every fold: test day absent from train/val, no mission in two sets, inner validation and training "
              f"contain all 5 classes | rule: {INNER_VAL_RULE} | fold hashes {FOLD_HASH}")

## 4. Models (references, verbatim)
- **AudioMamba:** v28's `UnimodalMamba` audio path. Encoder `Linear(80, 256) → LayerNorm → GELU → Linear`.
- **IMUMamba:** v28's `UnimodalMamba` sensor path. 1D-CNN encoder `Conv1d(6→64, k5) → BN → GELU → Conv1d(64→256,
  k3) → BN → GELU`.
- Both continue with: modality embedding + LayerNorm → **2 pre-norm residual real-Mamba blocks** (d_state 16,
  d_conv 2, expand 2; 874,496 params) → attention pooling → `Dropout(0.15) → Linear(256→128) → GELU → Linear(128→5)`.
- The models are **not changed**. CORAL reads the pooled embedding through a forward pre-hook on `classifier`, which
  sees the classifier's input.

In [ ]:
# ==============================================================================
# 6a. MODELS: AudioMamba (audio reference) + IMUMamba (IMU reference) — real mamba_ssm (HOOK 6)
# ==============================================================================
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda")

# E19: the v28 notebook is not read; the reference's transcribed settings are used unchanged.


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM (v28)."""

    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(D_MODEL)
        self.mamba = MAMBA_CLASS(d_model=D_MODEL, d_state=MAMBA_D_STATE, d_conv=MAMBA_D_CONV, expand=MAMBA_EXPAND)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


class AudioMamba(nn.Module):
    """v28 UnimodalMamba, audio path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention
    pooling -> classifier."""

    def __init__(self, in_dim=N_FEAT):
        super().__init__()
        self.audio_encoder = nn.Sequential(nn.Linear(in_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, x):
        h = self.token_norm(self.audio_encoder(x) + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1))


def verify_model(model):
    real = all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks) and MAMBA_CLASS.__module__.startswith("mamba_ssm")
    blocks_ok = len(model.mamba_blocks) == MAMBA_NUM_BLOCKS and all(
        (b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND) for b in model.mamba_blocks)
    n_mamba = sum(p.numel() for p in model.mamba_blocks.parameters())
    n_params = sum(p.numel() for p in model.parameters())
    return real and blocks_ok and n_mamba == EXPECTED_MAMBA_PARAMS, n_params, n_mamba


set_all_seeds(SEEDS[0])
_probe = AudioMamba().to(device)
_ok, N_PARAMS, _n_mamba = verify_model(_probe)
assert _ok, "model is not the expected real-mamba_ssm architecture"
print(f"AudioMamba: {N_PARAMS:,} parameters (Mamba blocks {_n_mamba:,} == {EXPECTED_MAMBA_PARAMS:,}) | block class "
      f"{MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | modules {[n for n, _ in _probe.named_children()]}")
with torch.no_grad():
    assert _probe(torch.zeros(2, TA, N_FEAT, device=device)).shape == (2, N_CLASSES)
hook(6, _ok, f"every block is mamba_ssm.Mamba (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); {N_PARAMS:,} parameters",
     stop=False)
del _probe
torch.cuda.empty_cache()


# ---- IMUMamba: IMU reference, verbatim (it uses the same MambaBlock wrapper around mamba_ssm.Mamba) ----
class IMUMamba(nn.Module):
    """v28 UnimodalMamba, sensor path (S1)."""
    def __init__(self, in_dim=6, n_classes=N_CLASSES):
        super().__init__()
        self.sensor_encoder=nn.Sequential(
            nn.Conv1d(in_dim,64,kernel_size=5,padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64,D_MODEL,kernel_size=3,padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.modality_embedding=nn.Embedding(1,D_MODEL)
        self.token_norm=nn.LayerNorm(D_MODEL)
        self.mamba_blocks=nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn=nn.Sequential(nn.Linear(D_MODEL,64),nn.Tanh(),nn.Linear(64,1))
        self.classifier=nn.Sequential(nn.Dropout(DROPOUT),nn.Linear(D_MODEL,D_MODEL//2),nn.GELU(),
                                      nn.Linear(D_MODEL//2,n_classes))

    def forward(self,x):
        z=self.sensor_encoder(x.transpose(1,2)).transpose(1,2)
        h=self.token_norm(z+self.modality_embedding.weight.view(1,1,-1))
        for b in self.mamba_blocks:
            h=b(h)
        a=F.softmax(self.pool_attn(h),dim=1)
        return self.classifier((h*a).sum(dim=1))


DEVICE = device                                                  # the IMU reference's name for the same device
set_all_seeds(SEEDS[0])
_probe = IMUMamba().to(DEVICE)
_ok_i, N_PARAMS_IMU, _n_mamba_i = verify_model(_probe)
with torch.no_grad():
    _shape_i = _probe.eval()(torch.zeros(2, TS, len(CHANNELS), device=DEVICE)).shape == (2, N_CLASSES)
hook(6, _ok_i and _shape_i, f"IMUMamba: every block is mamba_ssm.Mamba (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); "
                            f"{N_PARAMS_IMU:,} parameters (Mamba {_n_mamba_i:,})", stop=True)
N_PARAMS_AUDIO = N_PARAMS
del _probe
torch.cuda.empty_cache()

## 5. E19 training core
A single function trains every (modality, arm). With **arm A0** it follows the reference training exactly:
- `set_all_seeds(seed)` at the start of the run;
- train-only per-feature z-score; inverse-frequency class weights from the training windows;
- AdamW (lr 1e-4, weight decay 1e-4), cosine to 1e-6 over 20 epochs (stepped per epoch), grad clip 1.0, batch 32;
- a seeded shuffling DataLoader;
- validation macro-F1 after every epoch; checkpoint = the epoch with the strictly best validation macro-F1.

The other arms add exactly one thing each:

- **A1 AUG** (training batches only; the magnitudes are fixed a priori):
  - *IMU*, after z-scoring:
    - amplitude scale U(0.9, 1.1) per window and channel;
    - bias N(0, 0.1·σ) per window and channel;
    - noise N(0, 0.05·σ). σ is the training standard deviation, which is 1 after z-scoring.
  - *Audio*, before z-scoring, on the R0 4th-root features f:
    - a gain g ~ logU(0.8, 1.25) per window and channel, applied as f·g^0.5;
    - converted to power (f⁴), plus additive noise with mean 5% of that channel's training mean power
      (exponentially distributed, like a noise periodogram);
    - then the 4th root again.
    - The R0 token is the mean of 2 chunks' 4th-root features, so mapping it back to power is an approximation.
- **A2 CORAL:** loss = CE + 0.1 · mean over day pairs of ‖C_a − C_b‖²_F / (4 d²). C is the covariance of the pooled
  embedding (the classifier's input) for the windows of one training day in the batch. A day-balanced batch sampler
  draws 32 / n_days windows from every training day, with the same number of batches per epoch as A0. Day labels
  come from training missions only.
- **A3 DAYVAL:** within the fold's training pool (train + inner-validation missions), the held-out day = the largest
  training day if it has ≥ 15 missions, else the second largest (ties broken by `DAYS` order).
  - That day's missions are removed from fitting and form the validation set used for checkpoint selection.
  - The model is **not refit**.
  - The cost (missions removed from fitting) is logged.
- **A4 COMBO:** A1 + A2.

**Never used:** test-day statistics of any kind (z-score, gains and noise levels all come from fitting windows only);
augmentation at validation or test (asserted).

In [ ]:
# ==============================================================================
# 5a. E19 TRAINING CORE (new code; A0 == the reference training procedure)
# ==============================================================================
# mission aggregation: audio reference cell 6b, verbatim
def mission_level(missions, y, P):
    """Mission prediction = argmax of the mean window probabilities."""
    df = pd.DataFrame(P, columns=range(N_CLASSES))
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g.y.first().values, g[list(range(N_CLASSES))].mean().values, g.y.first().index.values


MODALITIES = {
    "imu": {"X": W, "y": y, "groups": groups, "days": days, "window_ids": window_ids, "phases": phases,
            "model_fn": lambda: IMUMamba(), "z_eps": SENSOR_STD_EPS, "n_params": N_PARAMS_IMU},
    "audio": {"X": X_ALL, "y": WMETA.cls.values.astype(np.int64), "groups": WMETA.mission.values, "days": WMETA.day.values,
              "window_ids": WMETA.window_id.values, "phases": WMETA.phase.values,
              "model_fn": lambda: AudioMamba(), "z_eps": 1e-6, "n_params": N_PARAMS_AUDIO},
}
for _m, _M in MODALITIES.items():
    assert len({len(v) for k, v in _M.items() if isinstance(v, np.ndarray)}) == 1, f"{_m}: misaligned window arrays"
AUG_GUARD = {"train_batches_augmented": 0, "eval_calls_with_aug": 0}
DAYMIX_LOG, A3_LOG, ID_HASHES = [], {}, {}


def split_for(arm, k):
    """(fit missions, validation missions, test missions, A3 info). A0/A1/A2/A4 use the reference inner split."""
    f = FOLDS[k]
    if arm != "A3":
        return list(f["train"]), list(f["val"]), list(f["test"]), {"heldout_day": "", "n_removed": 0}
    pool = list(f["train"]) + list(f["val"])
    cnt = Counter(MISSION_DAY[m] for m in pool)
    ranked = sorted(cnt, key=lambda d: (-cnt[d], DAYS.index(d)))
    dstar = ranked[0] if cnt[ranked[0]] >= A3_MIN_MISSIONS else ranked[1]
    fit = [m for m in f["train"] if MISSION_DAY[m] != dstar]
    val = [m for m in pool if MISSION_DAY[m] == dstar]
    return fit, val, list(f["test"]), {"heldout_day": dstar, "n_removed": len(f["train"]) - len(fit),
                                       "n_heldout_day_missions": cnt[dstar], "pool_day_counts": dict(cnt)}


class DayBalancedBatchSampler(torch.utils.data.Sampler):
    """Every batch draws BATCH_SIZE / n_days windows from each training day (remainder to the first days); each day's
    windows are cycled through seeded permutations. Same number of batches per epoch as the A0 loader."""

    def __init__(self, day_codes, batch_size, n_batches, seed):
        self.by_day = [np.where(day_codes == d)[0] for d in np.unique(day_codes)]
        nd = len(self.by_day)
        self.quota = [batch_size // nd + (1 if i < batch_size % nd else 0) for i in range(nd)]
        self.n_batches, self.rng = n_batches, np.random.default_rng(seed)

    def __iter__(self):
        pools = [list(self.rng.permutation(ix)) for ix in self.by_day]
        for _ in range(self.n_batches):
            batch = []
            for i, ix in enumerate(self.by_day):
                while len(pools[i]) < self.quota[i]:
                    pools[i] += list(self.rng.permutation(ix))
                batch += pools[i][:self.quota[i]]
                pools[i] = pools[i][self.quota[i]:]
            yield [int(b) for b in batch]

    def __len__(self):
        return self.n_batches


def coral_loss(z, d):
    """Mean over pairs of training days present in the batch of ||C_a - C_b||_F^2 / (4 D^2) (Deep CORAL)."""
    covs = []
    for dv in torch.unique(d):
        zz = z[d == dv]
        if len(zz) >= 2:
            zc = zz - zz.mean(dim=0, keepdim=True)
            covs.append(zc.T @ zc / (len(zz) - 1))
    if len(covs) < 2:
        return z.new_zeros(())
    D = z.shape[1]
    pairs = [((covs[i] - covs[j]) ** 2).sum() / (4 * D * D) for i in range(len(covs)) for j in range(i + 1, len(covs))]
    return torch.stack(pairs).mean()


def augment(mod, x, stats, gen, training):
    """A1 augmentation of ONE training batch. `training` must be True: augmentation never touches val/test."""
    assert training, "augmentation requested outside training"
    B = x.shape[0]
    if mod == "imu":                                   # x already z-scored: (B, TS, 6)
        C = x.shape[2]
        scale = IMU_AUG["scale_low"] + (IMU_AUG["scale_high"] - IMU_AUG["scale_low"]) * torch.rand(
            B, 1, C, generator=gen, device=x.device)
        bias = IMU_AUG["bias_sigma"] * torch.randn(B, 1, C, generator=gen, device=x.device)
        noise = IMU_AUG["noise_sigma"] * torch.randn(x.shape, generator=gen, device=x.device)
        out = x * scale + bias + noise
    else:                                              # x = raw R0 features (B, 64, 80), BEFORE z-scoring
        f = x.view(B, x.shape[1], N_CH, len(KEEP_BINS)).clamp_min(0)
        lo, hi = math.log(AUDIO_AUG["gain_low"]), math.log(AUDIO_AUG["gain_high"])
        g = torch.exp(lo + (hi - lo) * torch.rand(B, 1, N_CH, 1, generator=gen, device=x.device))
        P = (f * g.sqrt()) ** 4
        u = torch.rand(f.shape, generator=gen, device=x.device).clamp_min(1e-12)
        noise = AUDIO_AUG["noise_frac"] * stats["pbar"].view(1, 1, N_CH, 1) * (-torch.log(u))   # Exp(1) x 5% mean power
        out = ((P + noise) ** 0.25).reshape(x.shape)
    AUG_GUARD["train_batches_augmented"] += 1
    return out


@torch.no_grad()
def predict_mod(model, X, mu_t, sd_t, apply_aug=False, bs=256):
    """Validation / test prediction: z-score with TRAINING statistics, never augmented."""
    if apply_aug:
        AUG_GUARD["eval_calls_with_aug"] += 1
    assert not apply_aug, "augmentation must never be applied at validation/test"
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = (torch.from_numpy(X[i:i + bs]).to(DEVICE) - mu_t) / sd_t
        out.append(torch.softmax(model(xb), 1).float().cpu().numpy())
    return np.concatenate(out)


def run_arm(mod, arm, k, seed, epochs=EPOCHS, verbose=True):
    """One run = set_all_seeds(seed) FIRST, then fold-k fitting/validation split for the arm, train-only z-score and
    class weights, training (reference procedure + the arm's single change), test prediction."""
    t0 = time.time()
    set_all_seeds(seed)
    M = MODALITIES[mod]
    fit, val, test, a3 = split_for(arm, k)
    g = M["groups"]
    tr, va, te = np.isin(g, fit), np.isin(g, val), np.isin(g, test)
    # ---- integrity asserts (fail loudly) ----
    assert FOLDS[k]["day"] not in {MISSION_DAY[m] for m in fit + val}, "test day in fitting/validation"
    assert not (set(fit) & set(val)) and not ((set(fit) | set(val)) & set(test)), "mission in two sets"
    assert {MISSION_CLASS[m] for m in val} == set(range(N_CLASSES)), f"{arm} fold {k}: validation lacks a class"
    assert set(g[tr]) == set(fit), "z-score/class-weight windows must be exactly the fitting missions"
    id_hash = hashlib.sha1(("\n".join(M["window_ids"][te]) + "|" + "\n".join(sorted(test)) + "|" +
                            "\n".join(sorted(FOLDS[k]["train"] + FOLDS[k]["val"]))).encode()).hexdigest()[:16]
    ref_hash = ID_HASHES.setdefault((mod, k), id_hash)
    assert id_hash == ref_hash, f"{mod} fold {k} {arm}: test windows / missions differ from the other arms"
    if arm == "A3":
        A3_LOG[(mod, k)] = a3
        print(f"  A3 fold {k}: held-out TRAINING day {a3['heldout_day']} ({a3['n_heldout_day_missions']} missions) -> "
              f"validation; removed from fitting: {a3['n_removed']} of {len(FOLDS[k]['train'])} training missions "
              f"(training-pool day counts {a3['pool_day_counts']})")
    # ---- train-only statistics ----
    Xtr = M["X"][tr]
    F_ = Xtr.shape[-1]
    mu = Xtr.reshape(-1, F_).mean(0, dtype=np.float64).astype(np.float32)
    sd = (Xtr.reshape(-1, F_).std(0, dtype=np.float64) + M["z_eps"]).astype(np.float32)
    zhash = hashlib.sha1(mu.tobytes() + sd.tobytes()).hexdigest()[:12]
    mu_t, sd_t = torch.from_numpy(mu).to(DEVICE), torch.from_numpy(sd).to(DEVICE)
    stats = {}
    if mod == "audio":
        pw = Xtr.reshape(len(Xtr), Xtr.shape[1], N_CH, len(KEEP_BINS)).astype(np.float64) ** 4
        stats["pbar"] = torch.tensor(pw.mean(axis=(0, 1, 3)), dtype=torch.float32, device=DEVICE)   # (5,) train only
    ytr = M["y"][tr]
    day_codes = np.array([DAYS.index(d) for d in M["days"][tr]], dtype=np.int64)
    # ---- reference model / optimizer / loss ----
    model = M["model_fn"]().to(DEVICE)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()[:12]
    assert verify_model(model)[0], "not the expected real-mamba_ssm model"
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR_MIN)
    counts = np.bincount(ytr, minlength=N_CLASSES)
    weights = counts.sum() / (N_CLASSES * np.maximum(counts, 1))        # v28: inverse class frequency, fitting windows
    lossfn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
    ds = TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr), torch.from_numpy(day_codes))
    if arm in CORAL_ARMS:
        n_batches = math.ceil(len(ds) / BATCH_SIZE)
        loader = DataLoader(ds, batch_sampler=DayBalancedBatchSampler(day_codes, BATCH_SIZE, n_batches, seed),
                            num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=worker_init_fn)
    else:
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                            worker_init_fn=worker_init_fn, generator=make_generator(seed))
    aug_gen = torch.Generator(device=DEVICE)
    aug_gen.manual_seed(seed + 7919)
    emb = {}
    hk = model.classifier.register_forward_pre_hook(lambda mod_, inp: emb.__setitem__("z", inp[0])) \
        if arm in CORAL_ARMS else None
    Xva, yva = M["X"][va], M["y"][va]
    best_state, bestf, best_epoch, history = None, -1.0, None, []
    tag = f"[{mod} {arm} | fold {k} ({FOLDS[k]['day']}) | seed {seed}]"
    if verbose:
        print(f"{tag} init {init_sha1} | fit {len(fit)} missions / {int(tr.sum())} windows | val {len(val)} / "
              f"{int(va.sum())} | test {len(test)} / {int(te.sum())} | z {zhash}")
    for ep in range(1, epochs + 1):
        model.train()
        run_ce, run_coral, nb, mix = 0.0, 0.0, 0, None
        for xb, yb, db in loader:
            xb, yb, db = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True), db.to(DEVICE, non_blocking=True)
            if mix is None:
                mix = {DAYS[int(c)]: int(n) for c, n in zip(*np.unique(db.cpu().numpy(), return_counts=True))}
                if arm in CORAL_ARMS:
                    assert len(mix) >= 2, f"{tag}: CORAL batch with < 2 days {mix}"
                    DAYMIX_LOG.append((mod, arm, k, seed, ep, mix))
            if arm in AUG_ARMS and mod == "audio":
                xb = augment(mod, xb, stats, aug_gen, training=True)
            xb = (xb - mu_t) / sd_t
            if arm in AUG_ARMS and mod == "imu":
                xb = augment(mod, xb, stats, aug_gen, training=True)
            opt.zero_grad()
            ce = lossfn(model(xb), yb)
            loss = ce
            if arm in CORAL_ARMS:
                cl = coral_loss(emb["z"], db)
                loss = ce + CORAL_LAMBDA * cl
                run_coral += float(cl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            opt.step()
            run_ce += float(ce)
            nb += 1
        sched.step()
        assert math.isfinite(run_ce), f"{tag} non-finite training loss"
        p = predict_mod(model, Xva, mu_t, sd_t)
        f1v = f1_score(yva, p.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0)
        history.append((run_ce / nb, run_coral / nb, f1v))
        if f1v > bestf:
            bestf, best_epoch = f1v, ep
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"    epoch {ep:02d}/{epochs} | train CE {run_ce / nb:.4f}"
                  + (f" | CORAL {run_coral / nb:.5f}" if arm in CORAL_ARMS else "")
                  + f" | val macro-F1 {f1v:.4f}" + ("  * best" if ep == best_epoch else "")
                  + (f" | first-batch day mix {mix}" if arm in CORAL_ARMS else ""))
    if hk is not None:
        hk.remove()
    model.load_state_dict(best_state)
    P = predict_mod(model, M["X"][te], mu_t, sd_t)
    yt = M["y"][te]
    my_, mP_, _ = mission_level(M["groups"][te], yt, P)
    res = {"modality": mod, "arm": arm, "fold": k, "fold_day": FOLDS[k]["day"], "seed": seed,
           "window_F1": f1_score(yt, P.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0),
           "mission_F1": f1_score(my_, mP_.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0),
           "window_accuracy": float((P.argmax(1) == yt).mean()), "selected_epoch": best_epoch,
           "best_val_macro_f1": round(bestf, 6), "n_train_missions": len(fit), "n_val_missions": len(val),
           "n_test_missions": len(test), "n_train_windows": int(tr.sum()), "n_val_windows": int(va.sum()),
           "n_test_windows": int(te.sum()), "a3_heldout_day": a3["heldout_day"], "a3_missions_removed": a3["n_removed"],
           "inner_val_rule": INNER_VAL_RULE, "zstats_sha1": zhash, "init_sha1": init_sha1, "id_hash": id_hash,
           "n_params": M["n_params"], "epochs": epochs, "train_seconds": round(time.time() - t0, 1),
           "history": json.dumps([[round(a, 5), round(b, 6), round(c, 5)] for a, b, c in history])}
    np.savez(PROBS_DIR / f"{mod}_{arm}_{k}_{seed}.npz", window_id=M["window_ids"][te].astype(str),
             mission=M["groups"][te].astype(str), cls=yt, day=M["days"][te].astype(str),
             phase=M["phases"][te].astype(str), probs=P.astype(np.float32))
    if SAVE_CKPT:
        torch.save(best_state, CKPT_DIR / f"{mod}_{arm}_{k}_{seed}.pt")
    del model, opt, sched, loader, ds, best_state
    gc.collect()
    torch.cuda.empty_cache()
    return res

## 6. Pre-registered decision rule (fixed BEFORE the run cell; also written to `e19_rule.txt`)

```text
PRE-REGISTERED DECISION RULE (E19; fixed before any E19 result; applied in code)
Primary metric: pooled out-of-fold WINDOW macro-F1 from seed-averaged probabilities (95 missions, day-out).
Secondary: mission-level macro-F1 (argmax of the mean window probability), per-fold F1, worst-day F1.
Comparisons are made within ONE modality (IMU or audio-R0) and never pooled across modalities.
Candidates: A1 AUG, A2 CORAL, A3 DAYVAL, A4 COMBO (K = 4), each against A0 BASE of the same modality.
An arm is CREDITED for a modality only if ALL hold vs A0:
  1. pooled paired delta window macro-F1 >= +0.02 (seed noise floor ~0.016);
  2. the 99% paired mission-cluster bootstrap CI of the delta excludes 0 (2000 resamples of missions within class,
     seed 2028, the SAME resampled missions for both arms; 99% because K = 4);
  3. delta > 0 in >= 4 of 5 folds (seed-averaged, per held-out day);
  4. delta > 0 for all 3 seeds (each seed's own pooled out-of-fold F1);
  5. worst-day F1 does not regress: min over folds of the arm's seed-averaged fold F1 >= A0's.
WORSE (mirror): delta <= -0.02, the 99% CI lies entirely below 0, and delta < 0 in >= 4 of 5 folds.
Otherwise: NOT CREDITED. Every arm is reported whatever the outcome; a null result is a valid outcome.
Day gap closed (reported, not part of the rule): (arm - A0) / (random-split reference - A0), per modality.
```

In [ ]:
# ==============================================================================
# 6b. RUNS — modality-major (IMU first), then arm, seed, fold. CSV append after EVERY run; resumable on
#     (modality, arm, fold, seed); runtime estimate printed BEFORE training (nothing is reduced silently).
# ==============================================================================
CSV_FIELDS = ["modality", "arm", "fold", "fold_day", "seed", "window_F1", "mission_F1", "window_accuracy", "selected_epoch",
              "best_val_macro_f1", "n_train_missions", "n_val_missions", "n_test_missions", "n_train_windows",
              "n_val_windows", "n_test_windows", "a3_heldout_day", "a3_missions_removed", "inner_val_rule", "zstats_sha1",
              "init_sha1", "id_hash", "n_params", "epochs", "train_seconds", "history", "finished_at"]


def load_done():
    if not RESULTS_CSV.exists():
        return {}
    df = pd.read_csv(RESULTS_CSV)
    return {(r.modality, r.arm, int(r.fold), int(r.seed)): r for r in df.itertuples()
            if (PROBS_DIR / f"{r.modality}_{r.arm}_{int(r.fold)}_{int(r.seed)}.npz").exists()}


def append_row(res):
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**res, "finished_at": datetime.now().isoformat(timespec="seconds")})


PLAN = [(m, a, k, s) for m in MODS_TO_RUN for a in ARMS_TO_RUN for s in SEEDS_TO_RUN for k in FOLDS_TO_RUN]
_done = load_done()
_todo = [r for r in PLAN if r not in _done]
# ---- runtime estimate: reference seconds per 20-epoch run, scaled by epochs, CORAL overhead and A3's smaller fit set ----
_arm_factor = {"A0": 1.0, "A1": 1.05, "A2": 1.15, "A3": 0.75, "A4": 1.2}
_est = sum(REF_SECONDS_PER_RUN[m] * _arm_factor[a] * EPOCHS_TO_RUN / EPOCHS for m, a, _, _ in _todo)
print(f"{len(PLAN)} runs planned | already done: {len(PLAN) - len(_todo)} | to train now: {len(_todo)}")
print(f"ESTIMATED training time: {_est / 3600:.2f} h (+ parsing on a first session) — budget {BUDGET_HOURS} h")
if _est / 3600 > BUDGET_HOURS:
    loud(f"The estimate {_est / 3600:.2f} h exceeds the {BUDGET_HOURS} h budget. Nothing is reduced automatically:\n"
         "set RUN_SUBSET (top cell) to run part of the plan now; finished runs are skipped when you resume.")
_t0, _n = time.time(), 0
for (mod, arm, k, s) in PLAN:
    if (mod, arm, k, s) in _done:
        print(f"[{mod} {arm} fold {k} seed {s}] already in {RESULTS_CSV.name} — skipped")
        continue
    res = run_arm(mod, arm, k, s, epochs=EPOCHS_TO_RUN)
    append_row(res)
    _n += 1
    _el = time.time() - _t0
    print(f"[{_n}/{len(_todo)}] {mod} {arm} fold {k} ({res['fold_day']}) seed {s} | window F1 {res['window_F1']:.4f} | "
          f"mission F1 {res['mission_F1']:.4f} | epoch {res['selected_epoch']} | {res['train_seconds']:.0f} s | "
          f"ETA {(len(_todo) - _n) * _el / _n / 60:.0f} min")
print(f"\nTraining finished in {(time.time() - _t0) / 60:.1f} min | runs in CSV: {len(load_done())}/{len(PLAN)}")

# ---- run-level hooks 5, 7, 8, 9 (asserted per run above; summarised here) ----
_df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(columns=CSV_FIELDS)
_same = _df.groupby(["modality", "fold"]).id_hash.nunique().max() if len(_df) else 1
hook(5, _same == 1, f"test window ids + test missions + training pool identical across arms in every (modality, fold): "
                    f"max distinct hashes per (modality, fold) = {_same}", stop=True)
hook(7, AUG_GUARD["eval_calls_with_aug"] == 0, f"augmented training batches this session "
     f"{AUG_GUARD['train_batches_augmented']:,}; validation/test calls with augmentation {AUG_GUARD['eval_calls_with_aug']}",
     stop=True)
_mixes = [m for *_, m in DAYMIX_LOG]
hook(8, all(len(m) >= 2 for m in _mixes), f"{len(_mixes)} CORAL epochs logged this session; min days per first batch "
     f"{min((len(m) for m in _mixes), default='n/a')}; example {_mixes[0] if _mixes else 'n/a'}", stop=True)
_a3 = _df[_df.arm == "A3"].drop_duplicates(["modality", "fold"])
hook(9, True, "; ".join(f"{r.modality} fold {r.fold}: held-out {r.a3_heldout_day}, {r.a3_missions_removed} missions "
                        f"removed from fitting" for r in _a3.itertuples()) or "no A3 run", stop=False)

## 7. Results per modality (never pooled across modalities)
- All metrics are pooled out-of-fold from seed-averaged probabilities.
- Paired comparisons use the reference mission-cluster bootstrap (`cluster_bootstrap`, 2000 resamples, seed 2028).
  The pooled windows are in the same order for every arm, so both arms of a comparison get **identical** resampled
  missions.
- **Hook 10** (soft) compares A0's per-fold F1 with the reference runs. It prints WARN and never stops the notebook.

In [ ]:
# ==============================================================================
# 7a. RESULT HELPERS — audio reference functions (verbatim) + E19 pooling, paired comparison and the decision rule
# ==============================================================================
def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Resample MISSIONS within class (never windows). Returns bootstrap confusion matrices (B, 5, 5)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    cnt = np.zeros((len(ms), N_CLASSES), int)
    pos = {m: i for i, m in enumerate(ms)}
    mcls = np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        draw = rows[rng.integers(0, len(rows), (B, len(rows)))]
        cms[:, c, :] = cnt[draw].sum(1)
    return cms


def boot_summary(missions, y, pred):
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    p, r, f = prf_from_cm(cm)
    cms = cluster_bootstrap(missions, y, pred)
    bp, br, bf = zip(*(prf_from_cm(c) for c in cms))
    bp, br, bf = np.array(bp), np.array(br), np.array(bf)
    ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
    tab = pd.DataFrame({"precision": p, "P CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bp).T],
                        "recall": r, "R CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(br).T],
                        "f1": f, "F1 CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bf).T]},
                       index=[f"class {c}" for c in range(N_CLASSES)])
    macro = (f.mean(), *ci(bf.mean(1)))
    bal = (r.mean(), *ci(br.mean(1)))
    return tab, macro, bal, cm, (r, ci(br))


def macro_f1(yt, pred):
    return float(f1_score(yt, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0))


def pool_arm(mod, arm):
    """Seed-averaged out-of-fold probabilities in window-id order + per-seed probabilities."""
    per_seed, ids0, meta = {}, None, None
    for s in SEEDS:
        parts = [dict(np.load(PROBS_DIR / f"{mod}_{arm}_{k}_{s}.npz")) for k in range(len(DAYS))]
        ids = np.concatenate([p["window_id"] for p in parts])
        assert len(ids) == len(set(ids)) == len(MODALITIES[mod]["window_ids"]), f"{mod} {arm} seed {s}: windows"
        order = np.argsort(ids)
        per_seed[s] = np.concatenate([p["probs"] for p in parts])[order]
        if ids0 is None:
            ids0 = ids[order]
            meta = pd.DataFrame({k_: np.concatenate([p[k_] for p in parts])[order] for k_ in ("mission", "cls", "day")})
        assert np.array_equal(ids[order], ids0), "seeds cover different windows"
    return ids0, meta, per_seed


def arm_stats(meta, per_seed):
    yt = meta.cls.values.astype(int)
    P = np.mean(list(per_seed.values()), axis=0)
    pred = P.argmax(1)
    _, macro, bal, cm, (rec, _) = boot_summary(meta.mission.values, yt, pred)
    my_, mP_, mn_ = mission_level(meta.mission.values, yt, P)
    _, mmacro, _, _, _ = boot_summary(mn_, my_, mP_.argmax(1))
    fold_f1 = {d: macro_f1(yt[(meta.day == d).values], pred[(meta.day == d).values]) for d in DAYS}
    err = pred != yt
    return {"window_f1": macro, "mission_f1": mmacro, "recall": rec, "fold_f1": fold_f1, "min_fold": min(fold_f1.values()),
            "worst_day": min(fold_f1, key=fold_f1.get),
            "seed_f1": {s: macro_f1(yt, v.argmax(1)) for s, v in per_seed.items()},
            "adjacent_share": float((np.abs(pred - yt)[err] == 1).mean()) if err.any() else float("nan"),
            "faulty_called_normal": float((pred[yt > 0] == 0).mean()),
            "boot": np.array([prf_from_cm(c)[2].mean() for c in cluster_bootstrap(meta.mission.values, yt, pred)])}


def decision(delta, lo, hi, folds_pos, folds_neg, seeds_pos, n_seeds, min_fold, min_fold_ref):
    if (delta >= MIN_DELTA and lo > 0 and folds_pos >= MIN_FOLDS_SAME_SIGN and seeds_pos == n_seeds
            and min_fold >= min_fold_ref):
        return "CREDITED"
    if delta <= -MIN_DELTA and hi < 0 and folds_neg >= MIN_FOLDS_SAME_SIGN:
        return "WORSE"
    return "NOT CREDITED"


def paired_vs_a0(st, st0, mod):
    delta = st["window_f1"][0] - st0["window_f1"][0]
    q = (100 - PAIRED_CI) / 2
    lo, hi = np.percentile(st["boot"] - st0["boot"], [q, 100 - q])
    fd = {d: st["fold_f1"][d] - st0["fold_f1"][d] for d in DAYS}
    sd_ = {s: st["seed_f1"][s] - st0["seed_f1"][s] for s in st0["seed_f1"]}
    fp, fn = sum(v > 0 for v in fd.values()), sum(v < 0 for v in fd.values())
    sp = sum(v > 0 for v in sd_.values())
    gap = RANDOM_SPLIT_REF[mod] - st0["window_f1"][0]
    return {"delta": float(delta), "ci_lo": float(lo), "ci_hi": float(hi), "folds_pos": fp, "folds_neg": fn,
            "seeds_pos": sp, "fold_delta": fd, "seed_delta": sd_, "min_fold": st["min_fold"], "min_fold_a0": st0["min_fold"],
            "mission_delta": float(st["mission_f1"][0] - st0["mission_f1"][0]),
            "gap_closed": float(delta / gap) if gap > 0 else float("nan"),
            "verdict": decision(delta, lo, hi, fp, fn, sp, len(sd_), st["min_fold"], st0["min_fold"])}

In [ ]:
# ==============================================================================
# 7b. POOLED RESULTS, PAIRED TABLE (verdicts applied in code), PER-FOLD TABLE, SOFT HOOK 10
# ==============================================================================
_done = load_done()
COMPLETE = {m: [a for a in ARMS if all((m, a, k, s) in _done for k in range(len(DAYS)) for s in SEEDS)] for m in MODALITY_ORDER}
STATS, PAIRED = {}, {}
TABLE1 = TABLE2 = PERFOLD = None
if SMOKE:
    print("SMOKE run: pooled analysis skipped (it needs 5 folds x 3 seeds). Smoke rows:")
    print(_df[["modality", "arm", "fold", "seed", "window_F1", "mission_F1", "selected_epoch", "n_train_missions",
               "a3_heldout_day", "a3_missions_removed", "train_seconds"]].to_string(index=False))
else:
    _t1, _t2, _pf = [], [], []
    for mod in MODALITY_ORDER:
        if "A0" not in COMPLETE[mod]:
            loud(f"{mod}: A0 incomplete — no paired analysis for this modality yet (complete arms: {COMPLETE[mod]}).")
            continue
        ids_ref = None
        for arm in COMPLETE[mod]:
            ids, meta, per_seed = pool_arm(mod, arm)
            ids_ref = ids if ids_ref is None else ids_ref
            assert np.array_equal(ids, ids_ref), f"{mod} {arm}: different pooled windows — not paired"
            STATS[(mod, arm)] = arm_stats(meta, per_seed)
        _r = _df[(_df.modality == mod)]
        for arm in COMPLETE[mod]:
            st = STATS[(mod, arm)]
            rr = _r[_r.arm == arm]
            _t1.append({"modality": mod, "arm": arm, "name": ARM_NAMES[arm],
                        "window F1 [95% CI]": f"{st['window_f1'][0]:.4f} [{st['window_f1'][1]:.4f}, {st['window_f1'][2]:.4f}]",
                        "mission F1 [95% CI]": f"{st['mission_f1'][0]:.4f} [{st['mission_f1'][1]:.4f}, {st['mission_f1'][2]:.4f}]",
                        **{f"seed {s} F1": round(v, 4) for s, v in st["seed_f1"].items()},
                        "seed SD": round(float(np.std(list(st["seed_f1"].values()), ddof=1)), 4),
                        "worst day": f"{st['worst_day']} ({st['min_fold']:.4f})",
                        **{f"F1 {d}": round(v, 4) for d, v in st["fold_f1"].items()},
                        **{f"recall c{c}": round(float(st["recall"][c]), 3) for c in range(N_CLASSES)},
                        "adjacent-error share": round(st["adjacent_share"], 3),
                        "faulty called Normal": round(st["faulty_called_normal"], 4),
                        "mean fit missions": round(float(rr.n_train_missions.mean()), 1),
                        "mean selected epoch": round(float(rr.selected_epoch.mean()), 1)})
            for d in DAYS:
                _pf.append({"modality": mod, "arm": arm, "fold_day": d, "window_F1_seedavg": st["fold_f1"][d],
                            "delta_vs_A0": st["fold_f1"][d] - STATS[(mod, "A0")]["fold_f1"][d]})
        for arm in [a for a in CANDIDATE_ARMS if a in COMPLETE[mod]]:
            p = PAIRED[(mod, arm)] = paired_vs_a0(STATS[(mod, arm)], STATS[(mod, "A0")], mod)
            _t2.append({"modality": mod, "arm": arm, "name": ARM_NAMES[arm], "delta": round(p["delta"], 4),
                        "CI_lo_99": round(p["ci_lo"], 4), "CI_hi_99": round(p["ci_hi"], 4),
                        "folds_positive": p["folds_pos"], "seeds_positive": p["seeds_pos"],
                        "min_fold_F1": round(p["min_fold"], 4), "min_fold_F1_A0": round(p["min_fold_a0"], 4),
                        "delta_mission_F1": round(p["mission_delta"], 4), "day_gap_closed": round(p["gap_closed"], 3),
                        "verdict": p["verdict"]})
        for arm in [a for a in CANDIDATE_ARMS if a not in COMPLETE[mod]]:
            _t2.append({"modality": mod, "arm": arm, "name": ARM_NAMES[arm], "verdict": "NOT RUN / INCOMPLETE"})
        # ---- hook 10 (SOFT): A0 per-fold mean of per-run window F1 vs the reference runs ----
        a0 = _r[_r.arm == "A0"].groupby("fold").window_F1.mean().reindex(range(len(DAYS)))
        dev = (a0.values - np.array(REF_FOLD_F1[mod]))
        _ok10 = bool(np.all(np.abs(dev) <= REF_FOLD_TOL))
        hook(10, _ok10, f"{mod}: A0 per-fold F1 " + ", ".join(f"{DAYS[i]} {a0.values[i]:.4f} (ref {REF_FOLD_F1[mod][i]:.4f}, "
                                                            f"{dev[i]:+.4f})" for i in range(len(DAYS)))
             + ("" if _ok10 else f" — WARN: outside ±{REF_FOLD_TOL} on some fold (mamba-ssm CUDA non-determinism or a "
                                 f"different inner-validation rule; this run: {INNER_VAL_RULE})"), stop=False)
    TABLE1 = pd.DataFrame(_t1)
    TABLE2 = pd.DataFrame(_t2)
    PERFOLD = pd.DataFrame(_pf)
    TABLE1.to_csv(OUT / "e19_table_results.csv", index=False)
    TABLE2.to_csv(OUT / "e19_table_paired.csv", index=False)
    PERFOLD.to_csv(OUT / "e19_perfold.csv", index=False)
    pd.set_option("display.width", 250)
    for mod in MODALITY_ORDER:
        if len(TABLE1) and (TABLE1.modality == mod).any():
            print(f"\n=== {mod.upper()} — pooled out-of-fold results (seed-averaged) ===")
            print(TABLE1[TABLE1.modality == mod].set_index("arm").drop(columns="modality").T.to_string())
            print(f"\n=== {mod.upper()} — paired vs A0 ({PAIRED_CI:.0f}% CI, K = {len(CANDIDATE_ARMS)}) and VERDICT ===")
            print(TABLE2[TABLE2.modality == mod].set_index("arm").drop(columns="modality").to_string())

In [ ]:
# ==============================================================================
# 8. SUMMARY (generated from the numbers; verdict text comes from the rule in code) -> e19_summary.txt; zip; hooks
# ==============================================================================
import zipfile

_L = ["WHAT E19 TELLS US", ""]
if SMOKE:
    _L += ["SMOKE run only (fold 1, seed 42, 2 epochs, every arm and modality): checks that the pipeline runs end to end. "
           "No scientific conclusion can be drawn from it."]
else:
    for mod in MODALITY_ORDER:
        _L.append(f"[{mod.upper()}]")
        if (mod, "A0") not in STATS:
            _L += ["  A0 incomplete: no analysis for this modality.", ""]
            continue
        s0 = STATS[(mod, "A0")]
        _L.append(f"  A0 BASE: pooled window macro-F1 {s0['window_f1'][0]:.4f} [{s0['window_f1'][1]:.4f}, "
                  f"{s0['window_f1'][2]:.4f}], mission macro-F1 {s0['mission_f1'][0]:.4f}, worst day {s0['worst_day']} "
                  f"({s0['min_fold']:.4f}) | reference pooled {REF_POOLED_F1[mod]:.4f} | random-split reference "
                  f"{RANDOM_SPLIT_REF[mod]:.2f} -> day gap {RANDOM_SPLIT_REF[mod] - s0['window_f1'][0]:+.4f}")
        for arm in CANDIDATE_ARMS:
            if (mod, arm) not in PAIRED:
                _L.append(f"  {arm} {ARM_NAMES[arm]}: NOT RUN / INCOMPLETE")
                continue
            p = PAIRED[(mod, arm)]
            _L.append(f"  {arm} {ARM_NAMES[arm]}: delta {p['delta']:+.4f} [{p['ci_lo']:+.4f}, {p['ci_hi']:+.4f}] "
                      f"({PAIRED_CI:.0f}% paired CI), folds positive {p['folds_pos']}/5, seeds positive "
                      f"{p['seeds_pos']}/{len(SEEDS)}, worst-day F1 {p['min_fold']:.4f} vs A0 {p['min_fold_a0']:.4f}, "
                      f"mission delta {p['mission_delta']:+.4f}, day gap closed {100 * p['gap_closed']:+.0f}% -> {p['verdict']}")
        cred = [a for a in CANDIDATE_ARMS if (mod, a) in PAIRED and PAIRED[(mod, a)]["verdict"] == "CREDITED"]
        worse = [a for a in CANDIDATE_ARMS if (mod, a) in PAIRED and PAIRED[(mod, a)]["verdict"] == "WORSE"]
        _L.append("  -> " + (f"CREDITED for {mod}: {', '.join(cred)}." if cred else
                             f"No training-side arm is credited for {mod} by the pre-registered rule.")
                  + (f" WORSE: {', '.join(worse)}." if worse else ""))
        _a3 = _df[(_df.modality == mod) & (_df.arm == "A3")].drop_duplicates("fold")
        if len(_a3):
            _L.append("  A3 cost: " + "; ".join(f"fold {r.fold}: held out {r.a3_heldout_day}, {r.a3_missions_removed} "
                                                f"missions removed from fitting" for r in _a3.itertuples()))
        _L.append("")
_L += ["Limits",
       "- 95 missions (~19 per class); 5 unseen days, one of them (2021-12-22) with only 10 missions; 3 seeds.",
       f"- K = {len(CANDIDATE_ARMS)} candidate arms per modality (99% paired CIs); fixed magnitudes and lambda, never tuned.",
       "- The random-split reference (~0.89) is a stated project figure, not re-measured here; 'day gap closed' depends on it.",
       "- mamba-ssm CUDA kernels are not bit-deterministic; A0 vs reference is a soft check (hook 10).",
       "- The audio sample rate is unverified (per-channel FFT bins only); audio uses R0 only."]
SUMMARY = "\n".join(_L)
print(SUMMARY)
(OUT / ("e19_summary_smoke.txt" if SMOKE else "e19_summary.txt")).write_text(SUMMARY, encoding="utf-8")

_zip = OUT / ("e19_outputs_smoke.zip" if SMOKE else "e19_outputs.zip")
_n = 0
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("e19_*.csv")) + sorted(OUT.glob("e19_*.txt")) + sorted(PROBS_DIR.glob("*.npz")) \
            + sorted(FIG_DIR.glob("*.png")):
        zf.write(p, p.relative_to(OUT))
        _n += 1
print(f"\nWrote {_zip} ({_n} files, {_zip.stat().st_size / 1e6:.1f} MB; caches, checkpoints and the dataset excluded)")

print("\n" + "=" * 110)
print("VERIFICATION HOOKS (hook 10 is SOFT: WARN only)")
print("=" * 110)
for n in sorted(HOOK_NAMES):
    res = HOOKS.get(n, [])
    status = "NOT RUN" if not res else ("PASS" if all(ok for ok, _ in res) else ("WARN" if n == 10 else "FAIL"))
    print(f"  HOOK {n:<2} {status:<8} {HOOK_NAMES[n]}")
    for ok, msg in res:
        print(f"      {'PASS' if ok else ('WARN' if n == 10 else 'FAIL')}: {msg}")
print("=" * 110)
print("\n💾 Click \"Save Version\" -> \"Save & Run All (Commit)\" to keep /kaggle/working (or download e19_outputs.zip).")